In [ ]:
# | default_exp preprocessing.ocr

In [ ]:
%load_ext autoreload
%autoreload 2

# Document and image OCR preprocessing

> Recursively recognize PDF pages and PNG/JPEG images with local Ollama or Alibaba Cloud DashScope.

Each `.pdf`, `.png`, `.jpg`, or `.jpeg` source produces one UTF-8 Markdown document under a `.md` directory at the source root. Relative directories are preserved, PDF pages may run concurrently while their Markdown sections remain ordered, and multiple source files run concurrently through bounded asynchronous clients. Ollama remains the default provider; DashScope uses its OpenAI-compatible multimodal interface when selected.

`layout_mode="plain"` keeps the original one-request-per-page transcription path. The opt-in `layout_mode="pp-doclayout"` path uses the official GLM-OCR PP-DocLayout-V3 detector, recognizes each detected region with a task-specific prompt, preserves non-text crops in `<document>.assets/`, and writes `<document>.layout.json` as the retained layout representation. Install its optional runtime once with `uv sync --extra ocr-layout`; the detector model is downloaded on first use.

## Concurrency tuning

### Local Ollama interface (measured)

Measured on 2026-07-22 with local `glm-ocr:latest` on an RTX 4090 with 24 GiB VRAM, 123 GiB system RAM, and 32 logical CPUs. The 1.1B F16 model was loaded entirely on the GPU with a 4.8 GB runner footprint and a 32,768-token effective context. Ollama is configured with `OLLAMA_NUM_PARALLEL=4`, Flash Attention, and a `q4_0` KV cache.

A steady-state benchmark submitted eight distinct 200-DPI document pages through the same async image-chat interface used below:

| Request concurrency | Throughput | Mean latency |
| ---: | ---: | ---: |
| 1 | 0.924 pages/s | 1.08 s |
| 2 | 1.998 pages/s | 0.95 s |
| 4 | 3.508 pages/s | 0.94 s |
| 6 | 3.537 pages/s | 1.27 s |

Four concurrent requests are the throughput/latency optimum for the current local interface. Raising the client limit to six produced almost no additional throughput because the Ollama runner processes at most four requests; the extra requests wait in its queue and increase latency. Ollama also documents that parallel requests multiply context-memory requirements ([concurrency documentation](https://docs.ollama.com/faq#how-does-ollama-handle-concurrent-requests)).

### DashScope OpenAI-compatible interface (quota-based)

The current `.env` selects the China (Beijing) DashScope domain, `https://dashscope.aliyuncs.com/compatible-mode/v1`, and resolves to `qwen3.7-plus` because `OPENAILIKED_OCR_MODEL` is unset. Alibaba documents this domain as the Beijing DashScope endpoint; workspace-dedicated domains are recommended for production workloads that need higher, more stable concurrency ([regional domains](https://www.alibabacloud.com/help/en/model-studio/regions/)). No billable DashScope benchmark was run for this notebook, so the following values are conservative starting points rather than measured optima.

For `qwen3.7-plus` in the Chinese mainland, the published account-level limits are 30,000 requests/minute and 5,000,000 input-plus-output tokens/minute. Usage is aggregated across the root account, including its API keys, RAM users, and workspaces. The service may also enforce per-second RPS/TPS limits and dynamic traffic-burst protection, so a sudden batch can receive HTTP 429 even while below the minute totals ([rate limits](https://www.alibabacloud.com/help/en/model-studio/rate-limit), [rate-limit best practices](https://www.alibabacloud.com/help/en/model-studio/rate-limiting-best-practices)).

Each page is sent as one Base64 data URL. DashScope limits the encoded image to 10 MB, which this notebook enforces with PNG-to-JPEG fallback. Qwen3.7 accepts up to 16,777,216 pixels, but its default non-high-resolution policy uses `max_pixels=2,621,440`; larger pages may be downscaled. A typical 200-DPI A4 page is about 3.9 million pixels, so fine-print OCR may benefit from `vl_high_resolution_images=true` or a higher `max_pixels`, at the cost of more visual tokens, latency, and billed usage. Qwen3.7 uses approximately one visual token per `32 x 32` pixels ([visual-input limits](https://www.alibabacloud.com/help/en/model-studio/vision)). Plain mode retains the default policy; PP-DocLayout mode sends its smaller region crops with high-resolution vision enabled.

Start DashScope folder runs at `MAX_CONCURRENCY = 8` and `PAGE_CONCURRENCY = 2`. If monitoring shows no 429 responses, timeouts, or rising p95 latency, test 12 and then 16 global requests; retain an increase only when throughput improves materially. Reduce to 4/1 or 4/2 when burst errors occur. For one large PDF, 8/4 is a conservative starting point. The current implementation isolates failed files but does not automatically retry 429 responses; rerunning with `OVERWRITE = False` resumes missing outputs. Alibaba also supports the `X-DashScope-Wait-Timeout` header for burst queuing, but this notebook does not currently enable it.

### Recommended settings by interface

| Interface and workload | `MAX_CONCURRENCY` | `PAGE_CONCURRENCY` | Basis |
| --- | ---: | ---: | --- |
| Ollama, mixed recursive folder | 4 | 2 | Measured optimum on this RTX 4090 host |
| Ollama, one large PDF | 4 | 4 | Lets one document occupy all four runner slots |
| Ollama, four or more similar PDFs with fairness priority | 4 | 1 | One request per active document |
| DashScope, mixed recursive folder | 8 | 2 | Conservative unbilled starting point |
| DashScope, one large PDF | 8 | 4 | Conservative unbilled starting point |

In [ ]:
# | export
import asyncio
import base64
import json
import os
from collections import Counter
from dataclasses import dataclass
from html import escape
from pathlib import Path
from tempfile import NamedTemporaryFile, TemporaryDirectory
from time import perf_counter
from typing import Any, Callable, Literal, cast

import pymupdf
from dotenv import load_dotenv
from ollama import AsyncClient as AsyncOllamaClient
from ollama import Client as OllamaClient
from openai import AsyncOpenAI, OpenAI
from PIL import Image
from tqdm.auto import tqdm


In [ ]:
# | export
def _find_project_root() -> Path:
    """Find the nearest parent containing pyproject.toml."""
    starts: list[Path] = []
    module_file = globals().get("__file__")
    if isinstance(module_file, str):
        starts.append(Path(module_file).resolve().parent)
    starts.append(Path.cwd().resolve())
    for start in starts:
        for candidate in [start, *start.parents]:
            if (candidate / "pyproject.toml").is_file():
                return candidate
    return Path.cwd().resolve()


PROJ_ROOT = _find_project_root()
load_dotenv(PROJ_ROOT / ".env", override=False)

In [ ]:
# | export
OCRProvider = Literal["ollama", "dashscope"]
OCRLayoutMode = Literal["plain", "pp-doclayout"]
_OCRClient = OllamaClient | OpenAI
_AsyncOCRClient = AsyncOllamaClient | AsyncOpenAI


@dataclass(frozen=True)
class LayoutRegion:
    """One PP-DocLayout-V3 region in rendered-page pixel coordinates."""

    index: int
    label: str
    score: float
    bbox: tuple[int, int, int, int]
    task_type: Literal["text", "table", "formula", "figure"]


@dataclass(frozen=True)
class OCRResult:
    """Outcome of attempting to convert one source file to Markdown."""

    pdf_path: Path
    markdown_path: Path
    status: Literal["processed", "skipped", "failed"]
    pages_total: int = 0
    pages_completed: int = 0
    error: str | None = None

    @property
    def source_path(self) -> Path:
        """Source PDF or image path (preferred provider-neutral name)."""
        return self.pdf_path


In [ ]:
# | export
def _resolve_root(root_folder: Path | str) -> Path:
    root = Path(root_folder).expanduser().resolve()
    if not root.exists():
        raise FileNotFoundError(f"OCR root does not exist: {root}")
    if not root.is_dir():
        raise NotADirectoryError(f"OCR root is not a directory: {root}")
    return root


_SUPPORTED_SOURCE_SUFFIXES = frozenset({".pdf", ".png", ".jpg", ".jpeg"})
_IMAGE_SUFFIXES = frozenset({".png", ".jpg", ".jpeg"})


def _ocr_jobs(root: Path) -> list[tuple[Path, Path]]:
    """Return deterministic source/target pairs and reject target collisions."""
    output_root = root / ".md"
    source_files = [
        path
        for path in root.rglob("*")
        if path.is_file()
        and path.suffix.casefold() in _SUPPORTED_SOURCE_SUFFIXES
        and not path.is_relative_to(output_root)
    ]
    source_files.sort(
        key=lambda path: (
            path.relative_to(root).as_posix().casefold(),
            path.relative_to(root).as_posix(),
        )
    )

    jobs: list[tuple[Path, Path]] = []
    targets: dict[str, Path] = {}
    for source_path in source_files:
        relative_path = source_path.relative_to(root).with_suffix(".md")
        markdown_path = output_root / relative_path
        collision_key = markdown_path.as_posix().casefold()
        if previous := targets.get(collision_key):
            raise ValueError(
                f"OCR output collision: {previous} and {source_path} both map to {markdown_path}"
            )
        targets[collision_key] = source_path
        jobs.append((source_path, markdown_path))
    return jobs


In [ ]:
# | export
_DEFAULT_OLLAMA_MODEL = "glm-ocr"
_DEFAULT_DASHSCOPE_MODEL = "qwen3.7-plus"
_DEFAULT_DASHSCOPE_BASE_URL = "https://dashscope.aliyuncs.com/compatible-mode/v1"
_DEFAULT_LAYOUT_MODEL = "PaddlePaddle/PP-DocLayoutV3_safetensors"
_OLLAMA_PROMPT = "Text Recognition:"
_OLLAMA_LAYOUT_PROMPTS = {
    "text": "Text Recognition:",
    "table": "Table Recognition:",
    "formula": "Formula Recognition:",
    "figure": "Figure Recognition:",
}
_DASHSCOPE_PROMPT = (
    "Convert this document page to Markdown. Preserve the reading order, headings, "
    "paragraphs, lists, tables, code, and formulas. Return only the Markdown "
    "transcription without commentary."
)
_DASHSCOPE_LAYOUT_PROMPT = "qwenvl markdown"
_DASHSCOPE_MAX_DATA_URL_BYTES = 10 * 1024 * 1024
_LAYOUT_LABEL_TASKS = {
    "text": [
        "abstract",
        "algorithm",
        "aside_text",
        "content",
        "doc_title",
        "figure_title",
        "footer",
        "footnote",
        "formula_number",
        "header",
        "number",
        "paragraph_title",
        "reference",
        "reference_content",
        "seal",
        "text",
        "vertical_text",
        "vision_footnote",
    ],
    "table": ["table"],
    "formula": ["display_formula", "inline_formula"],
    "figure": ["chart", "footer_image", "header_image", "image"],
}
_LAYOUT_ASSET_TASKS = frozenset({"table", "formula", "figure"})


def _resolve_provider(provider: OCRProvider | str) -> OCRProvider:
    normalized = provider.strip().casefold()
    if normalized not in {"ollama", "dashscope"}:
        raise ValueError("provider must be 'ollama' or 'dashscope'")
    return cast(OCRProvider, normalized)


def _resolve_layout_mode(layout_mode: OCRLayoutMode | str) -> OCRLayoutMode:
    normalized = layout_mode.strip().casefold()
    if normalized not in {"plain", "pp-doclayout"}:
        raise ValueError("layout_mode must be 'plain' or 'pp-doclayout'")
    return cast(OCRLayoutMode, normalized)


def _resolve_model(provider: OCRProvider, model: str | None) -> str:
    if model is not None:
        if not model.strip():
            raise ValueError("model must not be empty")
        return model.strip()
    if provider == "dashscope":
        return (
            os.getenv("OPENAILIKED_OCR_MODEL", "").strip() or _DEFAULT_DASHSCOPE_MODEL
        )
    return _DEFAULT_OLLAMA_MODEL


def _resolve_prompt(provider: OCRProvider, prompt: str | None) -> str:
    if prompt is not None:
        if not prompt.strip():
            raise ValueError("prompt must not be empty")
        return prompt.strip()
    return _DASHSCOPE_PROMPT if provider == "dashscope" else _OLLAMA_PROMPT


def _resolve_layout_prompt(
    provider: OCRProvider,
    task_type: str,
    prompt: str | None,
) -> str:
    if prompt is not None:
        return _resolve_prompt(provider, prompt)
    if provider == "dashscope":
        return _DASHSCOPE_LAYOUT_PROMPT
    return _OLLAMA_LAYOUT_PROMPTS.get(task_type, _OLLAMA_PROMPT)


def create_pp_doclayout_detector(
    *,
    model_name: str = _DEFAULT_LAYOUT_MODEL,
    threshold: float = 0.3,
    device: str | None = None,
) -> Any:
    """Load and start the official GLM-OCR PP-DocLayout-V3 detector.

    Install the optional runtime with ``uv sync --extra ocr-layout``. The
    model is downloaded on first use unless ``model_name`` names a local path.
    """
    if not model_name.strip():
        raise ValueError("layout_model must not be empty")
    if not 0 <= threshold <= 1:
        raise ValueError("layout_threshold must be between 0 and 1")
    try:
        from glmocr.config import LayoutConfig
        from glmocr.layout import PPDocLayoutDetector
    except ImportError as error:
        raise RuntimeError(
            "PP-DocLayout-V3 requires the optional OCR layout runtime; "
            "run `uv sync --extra ocr-layout`"
        ) from error

    config = LayoutConfig(
        model_dir=model_name.strip(),
        threshold=threshold,
        batch_size=1,
        device=device,
        label_task_mapping=_LAYOUT_LABEL_TASKS,
    )
    detector = PPDocLayoutDetector(config)
    detector.start()
    return detector


def _response_content(response: object, provider: OCRProvider) -> str:
    if provider == "ollama":
        message = getattr(response, "message", None)
    else:
        choices = getattr(response, "choices", None)
        message = getattr(choices[0], "message", None) if choices else None
    content = getattr(message, "content", None)
    if not isinstance(content, str) or not content.strip():
        raise ValueError(f"{provider} OCR returned an empty response")
    return content.strip()


def _data_url(image_bytes: bytes, mime_type: str) -> str:
    encoded = base64.b64encode(image_bytes).decode("ascii")
    return f"data:{mime_type};base64,{encoded}"


def _dashscope_image_data_url(pixmap: pymupdf.Pixmap) -> str:
    png_data_url = _data_url(pixmap.tobytes("png"), "image/png")
    if len(png_data_url.encode("ascii")) <= _DASHSCOPE_MAX_DATA_URL_BYTES:
        return png_data_url

    jpeg_data_url = _data_url(
        pixmap.tobytes("jpeg", jpg_quality=90),
        "image/jpeg",
    )
    if len(jpeg_data_url.encode("ascii")) <= _DASHSCOPE_MAX_DATA_URL_BYTES:
        return jpeg_data_url
    raise ValueError(
        "Image payload exceeds DashScope's 10 MiB Base64 input limit; "
        "reduce PDF dpi or the source image dimensions"
    )


def _pixmap_markdown(
    pixmap: pymupdf.Pixmap,
    provenance: str,
    *,
    client: _OCRClient,
    provider: OCRProvider,
    model: str,
    prompt: str,
    high_resolution: bool = False,
) -> str:
    if provider == "ollama":
        response = client.chat(  # type: ignore[operator]
            model=model,
            messages=[
                {
                    "role": "user",
                    "content": prompt,
                    "images": [pixmap.tobytes("png")],
                }
            ],
            options={"temperature": 0},
        )
    else:
        extra_body: dict[str, bool] = {"enable_thinking": False}
        if high_resolution:
            extra_body["vl_high_resolution_images"] = True
        response = client.chat.completions.create(  # type: ignore[union-attr]
            model=model,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "image_url",
                            "image_url": {"url": _dashscope_image_data_url(pixmap)},
                        },
                        {"type": "text", "text": prompt},
                    ],
                }
            ],
            temperature=0,
            extra_body=extra_body,
        )
    content = _response_content(response, provider)
    return f"<!-- {provenance} -->\n\n{content}"


async def _pixmap_markdown_async(
    pixmap: pymupdf.Pixmap,
    provenance: str,
    *,
    client: _AsyncOCRClient,
    provider: OCRProvider,
    model: str,
    prompt: str,
    request_semaphore: asyncio.Semaphore | None = None,
    high_resolution: bool = False,
) -> str:
    async def send_request() -> object:
        if provider == "ollama":
            return await client.chat(  # type: ignore[union-attr]
                model=model,
                messages=[
                    {
                        "role": "user",
                        "content": prompt,
                        "images": [pixmap.tobytes("png")],
                    }
                ],
                options={"temperature": 0},
            )
        extra_body: dict[str, bool] = {"enable_thinking": False}
        if high_resolution:
            extra_body["vl_high_resolution_images"] = True
        return await client.chat.completions.create(  # type: ignore[union-attr]
            model=model,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "image_url",
                            "image_url": {"url": _dashscope_image_data_url(pixmap)},
                        },
                        {"type": "text", "text": prompt},
                    ],
                }
            ],
            temperature=0,
            extra_body=extra_body,
        )

    if request_semaphore is None:
        response = await send_request()
    else:
        async with request_semaphore:
            response = await send_request()
    content = _response_content(response, provider)
    return f"<!-- {provenance} -->\n\n{content}"


def _page_markdown(
    page: pymupdf.Page,
    page_number: int,
    *,
    client: _OCRClient,
    provider: OCRProvider,
    model: str,
    prompt: str,
    dpi: int,
) -> str:
    return _pixmap_markdown(
        page.get_pixmap(dpi=dpi, alpha=False),
        f"Page {page_number}",
        client=client,
        provider=provider,
        model=model,
        prompt=prompt,
    )


async def _page_markdown_async(
    page: pymupdf.Page,
    page_number: int,
    *,
    client: _AsyncOCRClient,
    provider: OCRProvider,
    model: str,
    prompt: str,
    dpi: int,
    request_semaphore: asyncio.Semaphore | None = None,
) -> str:
    return await _pixmap_markdown_async(
        page.get_pixmap(dpi=dpi, alpha=False),
        f"Page {page_number}",
        client=client,
        provider=provider,
        model=model,
        prompt=prompt,
        request_semaphore=request_semaphore,
    )


def _load_image_pixmap_with_pymupdf(path: Path) -> pymupdf.Pixmap:
    return pymupdf.Pixmap(str(path))


def _load_image_pixmap_with_pillow(path: Path) -> pymupdf.Pixmap:
    """Decode an image with Pillow and composite transparency onto white."""
    with Image.open(path) as image:
        image.load()
        rgba = image.convert("RGBA")
        background = Image.new("RGBA", rgba.size, (255, 255, 255, 255))
        rgb = Image.alpha_composite(background, rgba).convert("RGB")
        return pymupdf.Pixmap(
            pymupdf.csRGB, rgb.width, rgb.height, rgb.tobytes(), False
        )


def _load_image_pixmap(path: Path) -> pymupdf.Pixmap:
    try:
        pixmap = _load_image_pixmap_with_pymupdf(path)
    except Exception as pymupdf_error:
        try:
            return _load_image_pixmap_with_pillow(path)
        except Exception as pillow_error:
            raise ValueError(
                f"Could not decode image {path}: PyMuPDF: {pymupdf_error}; "
                f"Pillow: {pillow_error}"
            ) from pillow_error

    if pixmap.colorspace not in {pymupdf.csGRAY, pymupdf.csRGB}:
        pixmap = pymupdf.Pixmap(pymupdf.csRGB, pixmap)
    if pixmap.alpha:
        try:
            return _load_image_pixmap_with_pillow(path)
        except Exception:
            pixmap = pymupdf.Pixmap(pixmap, 0)
    return pixmap


def _pixmap_to_pil(pixmap: pymupdf.Pixmap) -> Image.Image:
    mode = "RGBA" if pixmap.alpha else ("L" if pixmap.n == 1 else "RGB")
    return Image.frombytes(mode, (pixmap.width, pixmap.height), pixmap.samples).convert(
        "RGB"
    )


def _pil_to_pixmap(image: Image.Image) -> pymupdf.Pixmap:
    rgb = image.convert("RGB")
    return pymupdf.Pixmap(pymupdf.csRGB, rgb.width, rgb.height, rgb.tobytes(), False)


def _layout_task_type(label: str, declared: object) -> str:
    if isinstance(declared, str) and declared in _LAYOUT_LABEL_TASKS:
        return cast(str, declared)
    for task_type, labels in _LAYOUT_LABEL_TASKS.items():
        if label in labels:
            return task_type
    return "text"


def _detect_layout_regions(
    detector: Any,
    image: Image.Image,
) -> list[LayoutRegion]:
    """Run a started GLM-OCR PPDocLayoutDetector for one rendered page."""
    if not hasattr(detector, "process"):
        raise TypeError("layout_detector must provide process(images, ...)")
    output = detector.process([image], save_visualization=False)
    pages = output[0] if isinstance(output, tuple) else output
    if not isinstance(pages, list) or len(pages) != 1:
        raise ValueError("PP-DocLayout returned an unexpected page result")
    raw_regions = pages[0]
    if not isinstance(raw_regions, list):
        raise ValueError("PP-DocLayout returned malformed regions")

    sortable: list[tuple[int, int, dict[str, Any]]] = []
    for position, raw_region in enumerate(raw_regions):
        if not isinstance(raw_region, dict):
            raise ValueError("PP-DocLayout returned a malformed region")
        declared_order = raw_region.get("index", position)
        order = declared_order if isinstance(declared_order, int) else position
        sortable.append((order, position, raw_region))
    sortable.sort(key=lambda item: (item[0], item[1]))

    regions: list[LayoutRegion] = []
    for reading_order, (_, _, raw_region) in enumerate(sortable, start=1):
        label = str(raw_region.get("label", "text")).strip() or "text"
        score = float(raw_region.get("score", 0.0))
        if "bbox_2d" in raw_region:
            raw_bbox = raw_region["bbox_2d"]
            if not isinstance(raw_bbox, (list, tuple)) or len(raw_bbox) != 4:
                raise ValueError("PP-DocLayout returned an invalid bbox_2d")
            x1, y1, x2, y2 = (
                float(raw_bbox[0]) * image.width / 1000,
                float(raw_bbox[1]) * image.height / 1000,
                float(raw_bbox[2]) * image.width / 1000,
                float(raw_bbox[3]) * image.height / 1000,
            )
        else:
            raw_bbox = raw_region.get("coordinate", raw_region.get("bbox"))
            if not isinstance(raw_bbox, (list, tuple)) or len(raw_bbox) != 4:
                raise ValueError(
                    "PP-DocLayout returned a region without a bounding box"
                )
            x1, y1, x2, y2 = (float(value) for value in raw_bbox)
        bbox = (
            max(0, min(image.width, int(round(x1)))),
            max(0, min(image.height, int(round(y1)))),
            max(0, min(image.width, int(round(x2)))),
            max(0, min(image.height, int(round(y2)))),
        )
        if bbox[2] <= bbox[0] or bbox[3] <= bbox[1]:
            continue
        task_type = _layout_task_type(label, raw_region.get("task_type"))
        regions.append(
            LayoutRegion(
                reading_order,
                label,
                score,
                bbox,
                cast(Literal["text", "table", "formula", "figure"], task_type),
            )
        )
    if not regions:
        regions.append(
            LayoutRegion(1, "text", 0.0, (0, 0, image.width, image.height), "text")
        )
    return regions


def _layout_asset_path(
    target: Path,
    page_number: int,
    region: LayoutRegion | None,
) -> Path:
    if region is None:
        filename = f"page-{page_number:04d}.png"
    else:
        label = (
            "".join(
                character if character.isalnum() else "-" for character in region.label
            ).strip("-")
            or "region"
        )
        filename = f"page-{page_number:04d}-region-{region.index:03d}-{label}.png"
    return Path(f"{target.stem}.assets") / filename


def _recognized_content(section: str) -> str:
    _, separator, content = section.partition("\n\n")
    return content.strip() if separator else section.strip()


def _render_layout_region(
    region: LayoutRegion,
    content: str,
    asset_path: Path | None,
) -> str:
    blocks: list[str] = []
    if asset_path is not None and region.task_type == "figure":
        alt = region.label.replace("_", " ").strip().title() or "Figure"
        blocks.append(f"![{alt}](<{asset_path.as_posix()}>)")

    normalized = content.strip()
    if region.label == "doc_title" and not normalized.startswith("#"):
        normalized = f"# {normalized}"
    elif region.label == "paragraph_title" and not normalized.startswith("#"):
        normalized = f"## {normalized}"
    elif region.label == "list" and not normalized.startswith(("- ", "* ", "1. ")):
        normalized = "\n".join(
            f"- {line.strip()}" for line in normalized.splitlines() if line.strip()
        )
    if normalized:
        blocks.append(normalized)

    provenance = f"Region {region.index}: {region.label}"
    return f"<!-- {provenance} -->\n\n" + "\n\n".join(blocks)


def _layout_page_sync(
    pixmap: pymupdf.Pixmap,
    page_number: int,
    *,
    target: Path,
    stage_assets: Path,
    detector: Any,
    client: _OCRClient,
    provider: OCRProvider,
    model: str,
    prompt: str | None,
    embed_page_image: bool,
) -> tuple[str, dict[str, Any]]:
    image = _pixmap_to_pil(pixmap)
    regions = _detect_layout_regions(detector, image)
    page_record: dict[str, Any] = {
        "page_number": page_number,
        "width": image.width,
        "height": image.height,
        "regions": [],
    }
    sections: list[str] = []
    if embed_page_image:
        page_asset = _layout_asset_path(target, page_number, None)
        page_file = stage_assets.parent / page_asset
        page_file.parent.mkdir(parents=True, exist_ok=True)
        page_file.write_bytes(pixmap.tobytes("png"))
        page_record["page_asset"] = page_asset.as_posix()
        sections.append(f"![Original page {page_number}](<{page_asset.as_posix()}>)")

    for region in regions:
        crop_image = image.crop(region.bbox)
        crop = _pil_to_pixmap(crop_image)
        region_prompt = _resolve_layout_prompt(provider, region.task_type, prompt)
        section = _pixmap_markdown(
            crop,
            f"Page {page_number} / Region {region.index}",
            client=client,
            provider=provider,
            model=model,
            prompt=region_prompt,
            high_resolution=True,
        )
        content = _recognized_content(section)
        asset_path: Path | None = None
        if region.task_type in _LAYOUT_ASSET_TASKS:
            asset_path = _layout_asset_path(target, page_number, region)
            asset_file = stage_assets.parent / asset_path
            asset_file.parent.mkdir(parents=True, exist_ok=True)
            asset_file.write_bytes(crop.tobytes("png"))
        sections.append(_render_layout_region(region, content, asset_path))
        page_record["regions"].append(
            {
                "index": region.index,
                "label": region.label,
                "score": region.score,
                "bbox": list(region.bbox),
                "task_type": region.task_type,
                "prompt": region_prompt,
                "asset": asset_path.as_posix() if asset_path is not None else None,
                "content": content,
            }
        )
    return (
        f"<!-- Page {page_number} -->\n\n" + "\n\n".join(sections),
        page_record,
    )


async def _layout_page_async(
    pixmap: pymupdf.Pixmap,
    page_number: int,
    *,
    target: Path,
    stage_assets: Path,
    detector: Any,
    client: _AsyncOCRClient,
    provider: OCRProvider,
    model: str,
    prompt: str | None,
    embed_page_image: bool,
    request_semaphore: asyncio.Semaphore | None,
    layout_semaphore: asyncio.Semaphore | None,
) -> tuple[str, dict[str, Any]]:
    image = _pixmap_to_pil(pixmap)
    if layout_semaphore is None:
        regions = await asyncio.to_thread(_detect_layout_regions, detector, image)
    else:
        async with layout_semaphore:
            regions = await asyncio.to_thread(_detect_layout_regions, detector, image)

    page_record: dict[str, Any] = {
        "page_number": page_number,
        "width": image.width,
        "height": image.height,
        "regions": [],
    }
    prefix_sections: list[str] = []
    if embed_page_image:
        page_asset = _layout_asset_path(target, page_number, None)
        page_file = stage_assets.parent / page_asset
        page_file.parent.mkdir(parents=True, exist_ok=True)
        page_file.write_bytes(pixmap.tobytes("png"))
        page_record["page_asset"] = page_asset.as_posix()
        prefix_sections.append(
            f"![Original page {page_number}](<{page_asset.as_posix()}>)"
        )

    prepared: list[tuple[LayoutRegion, pymupdf.Pixmap, str, Path | None]] = []
    for region in regions:
        crop = _pil_to_pixmap(image.crop(region.bbox))
        region_prompt = _resolve_layout_prompt(provider, region.task_type, prompt)
        asset_path: Path | None = None
        if region.task_type in _LAYOUT_ASSET_TASKS:
            asset_path = _layout_asset_path(target, page_number, region)
            asset_file = stage_assets.parent / asset_path
            asset_file.parent.mkdir(parents=True, exist_ok=True)
            asset_file.write_bytes(crop.tobytes("png"))
        prepared.append((region, crop, region_prompt, asset_path))

    async def recognize(
        prepared_region: tuple[LayoutRegion, pymupdf.Pixmap, str, Path | None],
    ) -> tuple[LayoutRegion, str, str, Path | None]:
        region, crop, region_prompt, asset_path = prepared_region
        section = await _pixmap_markdown_async(
            crop,
            f"Page {page_number} / Region {region.index}",
            client=client,
            provider=provider,
            model=model,
            prompt=region_prompt,
            request_semaphore=request_semaphore,
            high_resolution=True,
        )
        return region, _recognized_content(section), region_prompt, asset_path

    recognized = await asyncio.gather(*(recognize(item) for item in prepared))
    sections = list(prefix_sections)
    for region, content, region_prompt, asset_path in recognized:
        sections.append(_render_layout_region(region, content, asset_path))
        page_record["regions"].append(
            {
                "index": region.index,
                "label": region.label,
                "score": region.score,
                "bbox": list(region.bbox),
                "task_type": region.task_type,
                "prompt": region_prompt,
                "asset": asset_path.as_posix() if asset_path is not None else None,
                "content": content,
            }
        )
    return (
        f"<!-- Page {page_number} -->\n\n" + "\n\n".join(sections),
        page_record,
    )


def _publish_layout_bundle(
    stage_root: Path,
    target: Path,
    markdown: str,
    layout: dict[str, Any],
) -> None:
    """Publish assets and sidecar first, then atomically expose Markdown last."""
    stage_markdown = stage_root / target.name
    stage_sidecar = stage_root / f"{target.stem}.layout.json"
    stage_assets = stage_root / f"{target.stem}.assets"
    stage_assets.mkdir(parents=True, exist_ok=True)
    stage_markdown.write_text(markdown, encoding="utf-8", newline="\n")
    stage_sidecar.write_text(
        json.dumps(layout, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
        newline="\n",
    )

    final_sidecar = target.with_suffix(".layout.json")
    final_assets = target.parent / stage_assets.name
    publications = [
        (stage_assets, final_assets),
        (stage_sidecar, final_sidecar),
        (stage_markdown, target),
    ]
    backups: list[tuple[Path, Path]] = []
    published: list[tuple[Path, Path]] = []
    try:
        for backup_index, (_, final_path) in enumerate(publications):
            if final_path.exists():
                backup_path = stage_root / f"backup-{backup_index}"
                final_path.replace(backup_path)
                backups.append((backup_path, final_path))
        for staged_path, final_path in publications:
            staged_path.replace(final_path)
            published.append((final_path, staged_path))
    except Exception:
        for final_path, staged_path in reversed(published):
            if final_path.exists():
                final_path.replace(staged_path)
        for backup_path, final_path in reversed(backups):
            if backup_path.exists():
                backup_path.replace(final_path)
        raise


def _atomic_write_text(path: Path, content: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path: Path | None = None
    try:
        with NamedTemporaryFile(
            mode="w",
            encoding="utf-8",
            newline="\n",
            dir=path.parent,
            prefix=f".{path.name}.",
            suffix=".tmp",
            delete=False,
        ) as temporary_file:
            temporary_file.write(content)
            temporary_path = Path(temporary_file.name)
        temporary_path.replace(path)
        temporary_path = None
    finally:
        if temporary_path is not None:
            temporary_path.unlink(missing_ok=True)


In [ ]:
# | export
def ocr_pdf(
    pdf_path: Path | str,
    markdown_path: Path | str,
    *,
    client: _OCRClient,
    provider: OCRProvider = "ollama",
    model: str | None = None,
    prompt: str | None = None,
    dpi: int = 200,
    overwrite: bool = False,
    layout_mode: OCRLayoutMode = "plain",
    layout_detector: Any | None = None,
    layout_model: str = _DEFAULT_LAYOUT_MODEL,
    layout_threshold: float = 0.3,
    layout_device: str | None = None,
    embed_page_image: bool = False,
) -> OCRResult:
    """Convert one PDF to Markdown, optionally with PP-DocLayout-V3 regions.

    The final Markdown path is replaced only after every page succeeds. Exceptions
    are captured in the returned result so a folder batch can continue.
    """
    source = Path(pdf_path).expanduser().resolve()
    target = Path(markdown_path).expanduser().resolve()
    selected_provider = _resolve_provider(provider)
    selected_model = _resolve_model(selected_provider, model)
    selected_layout_mode = _resolve_layout_mode(layout_mode)
    selected_prompt = _resolve_prompt(selected_provider, prompt)
    if dpi <= 0:
        raise ValueError("dpi must be greater than zero")

    if target.exists():
        if not target.is_file():
            return OCRResult(
                source, target, "failed", error="Markdown target is not a file"
            )
        if not overwrite:
            return OCRResult(source, target, "skipped")

    pages_total = 0
    pages_completed = 0
    owned_detector: Any | None = None
    try:
        if not source.is_file():
            raise FileNotFoundError(f"PDF does not exist: {source}")
        detector = layout_detector
        if selected_layout_mode == "pp-doclayout" and detector is None:
            detector = create_pp_doclayout_detector(
                model_name=layout_model,
                threshold=layout_threshold,
                device=layout_device,
            )
            owned_detector = detector
        with pymupdf.open(source) as document:
            if document.needs_pass:
                raise ValueError("PDF requires a password")
            pages_total = document.page_count
            if pages_total == 0:
                raise ValueError("PDF contains no pages")

            if selected_layout_mode == "plain":
                page_sections: list[str] = []
                for page_number, page in enumerate(document, start=1):
                    page_sections.append(
                        _page_markdown(
                            page,
                            page_number,
                            client=client,
                            provider=selected_provider,
                            model=selected_model,
                            prompt=selected_prompt,
                            dpi=dpi,
                        )
                    )
                    pages_completed = page_number
                markdown = "\n\n".join(page_sections).rstrip() + "\n"
                _atomic_write_text(target, markdown)
            else:
                assert detector is not None
                target.parent.mkdir(parents=True, exist_ok=True)
                with TemporaryDirectory(
                    dir=target.parent, prefix=f".{target.stem}.layout-"
                ) as temporary_directory:
                    stage_root = Path(temporary_directory)
                    stage_assets = stage_root / f"{target.stem}.assets"
                    stage_assets.mkdir()
                    page_sections = []
                    page_records: list[dict[str, Any]] = []
                    for page_number, page in enumerate(document, start=1):
                        section, page_record = _layout_page_sync(
                            page.get_pixmap(dpi=dpi, alpha=False),
                            page_number,
                            target=target,
                            stage_assets=stage_assets,
                            detector=detector,
                            client=client,
                            provider=selected_provider,
                            model=selected_model,
                            prompt=prompt,
                            embed_page_image=embed_page_image,
                        )
                        page_sections.append(section)
                        page_records.append(page_record)
                        pages_completed = page_number
                    markdown = "\n\n".join(page_sections).rstrip() + "\n"
                    _publish_layout_bundle(
                        stage_root,
                        target,
                        markdown,
                        {
                            "schema_version": 1,
                            "source": str(source),
                            "provider": selected_provider,
                            "model": selected_model,
                            "layout_model": layout_model,
                            "dpi": dpi,
                            "pages": page_records,
                        },
                    )
        return OCRResult(source, target, "processed", pages_total, pages_completed)
    except Exception as error:
        return OCRResult(
            source,
            target,
            "failed",
            pages_total,
            pages_completed,
            f"{type(error).__name__}: {error}",
        )
    finally:
        if owned_detector is not None:
            try:
                owned_detector.stop()
            except Exception:
                pass


In [ ]:
# | export
def ocr_image(
    image_path: Path | str,
    markdown_path: Path | str,
    *,
    client: _OCRClient,
    provider: OCRProvider = "ollama",
    model: str | None = None,
    prompt: str | None = None,
    overwrite: bool = False,
    layout_mode: OCRLayoutMode = "plain",
    layout_detector: Any | None = None,
    layout_model: str = _DEFAULT_LAYOUT_MODEL,
    layout_threshold: float = 0.3,
    layout_device: str | None = None,
    embed_page_image: bool = False,
) -> OCRResult:
    """Convert one image to Markdown, optionally with PP-DocLayout-V3 regions."""
    source = Path(image_path).expanduser().resolve()
    target = Path(markdown_path).expanduser().resolve()
    selected_provider = _resolve_provider(provider)
    selected_model = _resolve_model(selected_provider, model)
    selected_layout_mode = _resolve_layout_mode(layout_mode)
    selected_prompt = _resolve_prompt(selected_provider, prompt)

    if target.exists():
        if not target.is_file():
            return OCRResult(
                source, target, "failed", error="Markdown target is not a file"
            )
        if not overwrite:
            return OCRResult(source, target, "skipped")

    pages_total = 0
    pages_completed = 0
    owned_detector: Any | None = None
    try:
        if not source.is_file():
            raise FileNotFoundError(f"Image does not exist: {source}")
        if source.suffix.casefold() not in _IMAGE_SUFFIXES:
            raise ValueError(f"Unsupported image type: {source.suffix or '<none>'}")

        pixmap = _load_image_pixmap(source)
        pages_total = 1
        if selected_layout_mode == "plain":
            markdown = (
                _pixmap_markdown(
                    pixmap,
                    "Image",
                    client=client,
                    provider=selected_provider,
                    model=selected_model,
                    prompt=selected_prompt,
                ).rstrip()
                + "\n"
            )
            _atomic_write_text(target, markdown)
        else:
            detector = layout_detector
            if detector is None:
                detector = create_pp_doclayout_detector(
                    model_name=layout_model,
                    threshold=layout_threshold,
                    device=layout_device,
                )
                owned_detector = detector
            target.parent.mkdir(parents=True, exist_ok=True)
            with TemporaryDirectory(
                dir=target.parent, prefix=f".{target.stem}.layout-"
            ) as temporary_directory:
                stage_root = Path(temporary_directory)
                stage_assets = stage_root / f"{target.stem}.assets"
                stage_assets.mkdir()
                markdown, page_record = _layout_page_sync(
                    pixmap,
                    1,
                    target=target,
                    stage_assets=stage_assets,
                    detector=detector,
                    client=client,
                    provider=selected_provider,
                    model=selected_model,
                    prompt=prompt,
                    embed_page_image=embed_page_image,
                )
                _publish_layout_bundle(
                    stage_root,
                    target,
                    markdown.rstrip() + "\n",
                    {
                        "schema_version": 1,
                        "source": str(source),
                        "provider": selected_provider,
                        "model": selected_model,
                        "layout_model": layout_model,
                        "dpi": None,
                        "pages": [page_record],
                    },
                )
        pages_completed = 1
        return OCRResult(source, target, "processed", pages_total, pages_completed)
    except Exception as error:
        return OCRResult(
            source,
            target,
            "failed",
            pages_total,
            pages_completed,
            f"{type(error).__name__}: {error}",
        )
    finally:
        if owned_detector is not None:
            try:
                owned_detector.stop()
            except Exception:
                pass


In [ ]:
# | export
async def _ocr_pdf_async(
    pdf_path: Path | str,
    markdown_path: Path | str,
    *,
    client: _AsyncOCRClient,
    provider: OCRProvider = "ollama",
    model: str | None = None,
    prompt: str | None = None,
    dpi: int = 200,
    overwrite: bool = False,
    page_concurrency: int = 1,
    request_semaphore: asyncio.Semaphore | None = None,
    layout_mode: OCRLayoutMode = "plain",
    layout_detector: Any | None = None,
    layout_model: str = _DEFAULT_LAYOUT_MODEL,
    layout_threshold: float = 0.3,
    layout_device: str | None = None,
    embed_page_image: bool = False,
    layout_semaphore: asyncio.Semaphore | None = None,
    page_started: Callable[[int], None] | None = None,
    page_progress: Callable[[int, int, float], None] | None = None,
) -> OCRResult:
    """Asynchronously convert one PDF while preserving page order."""
    source = Path(pdf_path).expanduser().resolve()
    target = Path(markdown_path).expanduser().resolve()
    selected_provider = _resolve_provider(provider)
    selected_model = _resolve_model(selected_provider, model)
    selected_layout_mode = _resolve_layout_mode(layout_mode)
    selected_prompt = _resolve_prompt(selected_provider, prompt)
    if dpi <= 0:
        raise ValueError("dpi must be greater than zero")
    if page_concurrency <= 0:
        raise ValueError("page_concurrency must be greater than zero")

    if target.exists():
        if not target.is_file():
            return OCRResult(
                source, target, "failed", error="Markdown target is not a file"
            )
        if not overwrite:
            return OCRResult(source, target, "skipped")

    pages_total = 0
    pages_completed = 0
    owned_detector: Any | None = None
    layout_temporary_directory: TemporaryDirectory[str] | None = None
    try:
        if not source.is_file():
            raise FileNotFoundError(f"PDF does not exist: {source}")
        detector = layout_detector
        stage_root: Path | None = None
        stage_assets: Path | None = None
        if selected_layout_mode == "pp-doclayout":
            if detector is None:
                detector = await asyncio.to_thread(
                    create_pp_doclayout_detector,
                    model_name=layout_model,
                    threshold=layout_threshold,
                    device=layout_device,
                )
                owned_detector = detector
            target.parent.mkdir(parents=True, exist_ok=True)
            layout_temporary_directory = TemporaryDirectory(
                dir=target.parent, prefix=f".{target.stem}.layout-"
            )
            stage_root = Path(layout_temporary_directory.name)
            stage_assets = stage_root / f"{target.stem}.assets"
            stage_assets.mkdir()
        with pymupdf.open(source) as document:
            if document.needs_pass:
                raise ValueError("PDF requires a password")
            pages_total = document.page_count
            if pages_total == 0:
                raise ValueError("PDF contains no pages")
            if page_started is not None:
                page_started(pages_total)

            page_sections = [""] * pages_total
            page_records: list[dict[str, Any] | None] = [None] * pages_total
            page_tasks: set[
                asyncio.Task[tuple[int, str, dict[str, Any] | None, float]]
            ] = set()
            pending: set[
                asyncio.Task[tuple[int, str, dict[str, Any] | None, float]]
            ] = set()
            next_page_number = 1

            async def process_page(
                page_number: int,
            ) -> tuple[int, str, dict[str, Any] | None, float]:
                page_started_at = perf_counter()
                page = document.load_page(page_number - 1)
                if selected_layout_mode == "plain":
                    section = await _page_markdown_async(
                        page,
                        page_number,
                        client=client,
                        provider=selected_provider,
                        model=selected_model,
                        prompt=selected_prompt,
                        dpi=dpi,
                        request_semaphore=request_semaphore,
                    )
                    page_record = None
                else:
                    assert detector is not None and stage_assets is not None
                    section, page_record = await _layout_page_async(
                        page.get_pixmap(dpi=dpi, alpha=False),
                        page_number,
                        target=target,
                        stage_assets=stage_assets,
                        detector=detector,
                        client=client,
                        provider=selected_provider,
                        model=selected_model,
                        prompt=prompt,
                        embed_page_image=embed_page_image,
                        request_semaphore=request_semaphore,
                        layout_semaphore=layout_semaphore,
                    )
                return (
                    page_number,
                    section,
                    page_record,
                    perf_counter() - page_started_at,
                )

            def schedule_next_page() -> None:
                nonlocal next_page_number
                task = asyncio.create_task(process_page(next_page_number))
                page_tasks.add(task)
                pending.add(task)
                next_page_number += 1

            for _ in range(min(page_concurrency, pages_total)):
                schedule_next_page()

            try:
                while pending:
                    completed, pending = await asyncio.wait(
                        pending, return_when=asyncio.FIRST_COMPLETED
                    )
                    for page_task in completed:
                        page_number, section, page_record, elapsed_s = (
                            page_task.result()
                        )
                        page_sections[page_number - 1] = section
                        page_records[page_number - 1] = page_record
                        pages_completed += 1
                        if page_progress is not None:
                            page_progress(pages_completed, pages_total, elapsed_s)
                        if next_page_number <= pages_total:
                            schedule_next_page()
            finally:
                for page_task in page_tasks:
                    if not page_task.done():
                        page_task.cancel()
                await asyncio.gather(*page_tasks, return_exceptions=True)

        markdown = "\n\n".join(page_sections).rstrip() + "\n"
        if selected_layout_mode == "plain":
            _atomic_write_text(target, markdown)
        else:
            assert stage_root is not None
            _publish_layout_bundle(
                stage_root,
                target,
                markdown,
                {
                    "schema_version": 1,
                    "source": str(source),
                    "provider": selected_provider,
                    "model": selected_model,
                    "layout_model": layout_model,
                    "dpi": dpi,
                    "pages": [record for record in page_records if record is not None],
                },
            )
        return OCRResult(source, target, "processed", pages_total, pages_completed)
    except Exception as error:
        return OCRResult(
            source,
            target,
            "failed",
            pages_total,
            pages_completed,
            f"{type(error).__name__}: {error}",
        )
    finally:
        if layout_temporary_directory is not None:
            layout_temporary_directory.cleanup()
        if owned_detector is not None:
            try:
                await asyncio.to_thread(owned_detector.stop)
            except Exception:
                pass


async def _ocr_image_async(
    image_path: Path | str,
    markdown_path: Path | str,
    *,
    client: _AsyncOCRClient,
    provider: OCRProvider = "ollama",
    model: str | None = None,
    prompt: str | None = None,
    overwrite: bool = False,
    request_semaphore: asyncio.Semaphore | None = None,
    layout_mode: OCRLayoutMode = "plain",
    layout_detector: Any | None = None,
    layout_model: str = _DEFAULT_LAYOUT_MODEL,
    layout_threshold: float = 0.3,
    layout_device: str | None = None,
    embed_page_image: bool = False,
    layout_semaphore: asyncio.Semaphore | None = None,
) -> OCRResult:
    """Asynchronously convert one PNG or JPEG image."""
    source = Path(image_path).expanduser().resolve()
    target = Path(markdown_path).expanduser().resolve()
    selected_provider = _resolve_provider(provider)
    selected_model = _resolve_model(selected_provider, model)
    selected_layout_mode = _resolve_layout_mode(layout_mode)
    selected_prompt = _resolve_prompt(selected_provider, prompt)

    if target.exists():
        if not target.is_file():
            return OCRResult(
                source, target, "failed", error="Markdown target is not a file"
            )
        if not overwrite:
            return OCRResult(source, target, "skipped")

    pages_total = 0
    pages_completed = 0
    owned_detector: Any | None = None
    try:
        if not source.is_file():
            raise FileNotFoundError(f"Image does not exist: {source}")
        if source.suffix.casefold() not in _IMAGE_SUFFIXES:
            raise ValueError(f"Unsupported image type: {source.suffix or '<none>'}")

        pixmap = _load_image_pixmap(source)
        pages_total = 1
        if selected_layout_mode == "plain":
            markdown = (
                await _pixmap_markdown_async(
                    pixmap,
                    "Image",
                    client=client,
                    provider=selected_provider,
                    model=selected_model,
                    prompt=selected_prompt,
                    request_semaphore=request_semaphore,
                )
            ).rstrip() + "\n"
            _atomic_write_text(target, markdown)
        else:
            detector = layout_detector
            if detector is None:
                detector = await asyncio.to_thread(
                    create_pp_doclayout_detector,
                    model_name=layout_model,
                    threshold=layout_threshold,
                    device=layout_device,
                )
                owned_detector = detector
            target.parent.mkdir(parents=True, exist_ok=True)
            with TemporaryDirectory(
                dir=target.parent, prefix=f".{target.stem}.layout-"
            ) as temporary_directory:
                stage_root = Path(temporary_directory)
                stage_assets = stage_root / f"{target.stem}.assets"
                stage_assets.mkdir()
                markdown, page_record = await _layout_page_async(
                    pixmap,
                    1,
                    target=target,
                    stage_assets=stage_assets,
                    detector=detector,
                    client=client,
                    provider=selected_provider,
                    model=selected_model,
                    prompt=prompt,
                    embed_page_image=embed_page_image,
                    request_semaphore=request_semaphore,
                    layout_semaphore=layout_semaphore,
                )
                _publish_layout_bundle(
                    stage_root,
                    target,
                    markdown.rstrip() + "\n",
                    {
                        "schema_version": 1,
                        "source": str(source),
                        "provider": selected_provider,
                        "model": selected_model,
                        "layout_model": layout_model,
                        "dpi": None,
                        "pages": [page_record],
                    },
                )
        pages_completed = 1
        return OCRResult(source, target, "processed", pages_total, pages_completed)
    except Exception as error:
        return OCRResult(
            source,
            target,
            "failed",
            pages_total,
            pages_completed,
            f"{type(error).__name__}: {error}",
        )
    finally:
        if owned_detector is not None:
            try:
                await asyncio.to_thread(owned_detector.stop)
            except Exception:
                pass


In [ ]:
# | export
def _create_async_ocr_client(
    provider: OCRProvider,
    *,
    host: str,
    request_timeout_s: float,
) -> _AsyncOCRClient:
    if provider == "ollama":
        if not host.strip():
            raise ValueError("host must not be empty")
        return AsyncOllamaClient(host=host, timeout=request_timeout_s)

    api_key = os.getenv("DASHSCOPE_API_KEY", "").strip()
    if not api_key:
        raise RuntimeError(
            f"DASHSCOPE_API_KEY is not configured in {PROJ_ROOT / '.env'}"
        )
    base_url = os.getenv("DASHSCOPE_API_URL", "").strip() or _DEFAULT_DASHSCOPE_BASE_URL
    if (
        not base_url.startswith(("http://", "https://"))
        or "/compatible-mode/" not in base_url
    ):
        raise RuntimeError(
            "DASHSCOPE_API_URL must be an OpenAI-compatible HTTP(S) endpoint"
        )
    return AsyncOpenAI(api_key=api_key, base_url=base_url, timeout=request_timeout_s)


@dataclass
class _ActivePageProgress:
    source_path: Path
    pages_total: int | None = None
    pages_completed: int = 0
    last_page_elapsed_s: float | None = None


def _running_in_notebook() -> bool:
    """Return whether output is being rendered by a Jupyter kernel."""
    try:
        from IPython import get_ipython
    except ImportError:
        return False
    shell = get_ipython()
    return shell is not None and shell.__class__.__name__ == "ZMQInteractiveShell"


class _OCRFolderProgress:
    """Render page and file progress reliably in terminals and notebooks."""

    def __init__(self, total_files: int, description: str, show_pages: bool) -> None:
        self._total_files = total_files
        self._files_completed = 0
        self._description = description
        self._show_pages = show_pages
        self._active_order: list[int] = []
        self._active: dict[int, _ActivePageProgress] = {}
        self._displayed_index: int | None = None
        self._notebook_mode = _running_in_notebook()
        self._html_factory: Callable[[str], Any] | None = None
        self._display: Callable[..., Any] | None = None
        self._display_handle: Any = None
        self._page_bar: Any = None
        self._file_bar: Any = None

        if self._notebook_mode:
            from IPython.display import HTML, display

            self._html_factory = HTML
            self._display = display
            self._refresh_notebook()
        else:
            self._page_bar = (
                tqdm(
                    total=None,
                    desc="OCR pages: waiting",
                    unit="page",
                    position=0,
                    leave=False,
                    dynamic_ncols=True,
                )
                if show_pages
                else None
            )
            self._file_bar = tqdm(
                total=total_files,
                desc=description,
                unit="file",
                position=1 if show_pages else 0,
                leave=True,
                dynamic_ncols=True,
            )

    def start_file(
        self, index: int, source_path: Path, pages_total: int | None = None
    ) -> None:
        self._active[index] = _ActivePageProgress(source_path, pages_total)
        self._active_order.append(index)
        self._refresh_page_bar()

    def set_pages_total(self, index: int, pages_total: int) -> None:
        state = self._active.get(index)
        if state is not None:
            state.pages_total = pages_total
            self._refresh_page_bar()

    def complete_page(
        self, index: int, pages_completed: int, pages_total: int, elapsed_s: float
    ) -> None:
        state = self._active.get(index)
        if state is not None:
            state.pages_total = pages_total
            state.pages_completed = pages_completed
            state.last_page_elapsed_s = elapsed_s
            self._refresh_page_bar()

    def finish_file(self, index: int) -> None:
        self._active.pop(index, None)
        if index in self._active_order:
            self._active_order.remove(index)
        self._files_completed += 1
        if self._file_bar is not None:
            self._file_bar.update(1)
        self._refresh_page_bar()

    def _refresh_page_bar(self) -> None:
        if self._notebook_mode:
            self._refresh_notebook()
            return
        if self._page_bar is None:
            return
        if not self._active_order:
            self._displayed_index = None
            self._page_bar.clear()
            self._page_bar.set_description_str("OCR pages: waiting", refresh=False)
            self._page_bar.total = None
            self._page_bar.n = 0
            self._page_bar.set_postfix_str("", refresh=False)
            self._page_bar.refresh()
            return

        newest_index = self._active_order[-1]
        state = self._active[newest_index]
        if newest_index != self._displayed_index:
            self._page_bar.clear()
            self._displayed_index = newest_index
            self._page_bar.n = 0
            self._page_bar.last_print_n = 0
            now = self._page_bar._time()
            self._page_bar.start_t = now
            self._page_bar.last_print_t = now
        self._page_bar.set_description_str(
            f"OCR pages: {state.source_path}", refresh=False
        )
        self._page_bar.total = state.pages_total
        self._page_bar.n = state.pages_completed
        elapsed = state.last_page_elapsed_s
        self._page_bar.set_postfix_str(
            f"last={elapsed:.2f}s" if elapsed is not None else "",
            refresh=False,
        )
        self._page_bar.refresh()

    @staticmethod
    def _html_progress_row(
        label: str, completed: int, total: int | None, detail: str = ""
    ) -> str:
        count = f"{completed}/{total}" if total is not None else f"{completed}/?"
        percentage = 100 * completed / total if total else 0
        progress = (
            f'<progress value="{min(completed, total)}" max="{total}" '
            'style="width:100%;height:0.8rem"></progress>'
            if total is not None
            else '<progress style="width:100%;height:0.8rem"></progress>'
        )
        suffix = f" &nbsp; {escape(detail)}" if detail else ""
        return (
            '<div style="margin:0 0 0.45rem 0">'
            '<div style="display:flex;gap:1rem;justify-content:space-between;"'
            ">"
            f'<span style="overflow-wrap:anywhere">{escape(label)}</span>'
            f'<span style="white-space:nowrap">{percentage:.0f}% &nbsp; '
            f"{count}{suffix}</span></div>{progress}</div>"
        )

    def _notebook_html(self) -> str:
        rows: list[str] = []
        if self._show_pages:
            if self._active_order:
                state = self._active[self._active_order[-1]]
                elapsed = state.last_page_elapsed_s
                detail = f"last={elapsed:.2f}s" if elapsed is not None else ""
                rows.append(
                    self._html_progress_row(
                        f"OCR pages: {state.source_path}",
                        state.pages_completed,
                        state.pages_total,
                        detail,
                    )
                )
            else:
                rows.append(self._html_progress_row("OCR pages: waiting", 0, None))
        rows.append(
            self._html_progress_row(
                self._description, self._files_completed, self._total_files
            )
        )
        return (
            '<div style="font-family:var(--vscode-editor-font-family,monospace);'
            'font-size:var(--vscode-editor-font-size,13px);padding:0.25rem 0">'
            + "".join(rows)
            + "</div>"
        )

    def _refresh_notebook(self) -> None:
        if self._html_factory is None or self._display is None:
            return
        content = self._html_factory(self._notebook_html())
        if self._display_handle is None:
            self._display_handle = self._display(content, display_id=True)
        else:
            self._display_handle.update(content)

    def close(self) -> None:
        if self._notebook_mode:
            self._refresh_notebook()
            return
        if self._page_bar is not None:
            self._page_bar.close()
        if self._file_bar is not None:
            self._file_bar.close()


async def ocr_folder(
    root_folder: Path | str,
    *,
    provider: OCRProvider = "ollama",
    host: str = "http://127.0.0.1:11434",
    model: str | None = None,
    prompt: str | None = None,
    dpi: int = 200,
    overwrite: bool = False,
    request_timeout_s: float = 180,
    max_concurrency: int = 4,
    page_concurrency: int = 1,
    show_page_progress: bool = True,
    client: _AsyncOCRClient | None = None,
    layout_mode: OCRLayoutMode = "plain",
    layout_detector: Any | None = None,
    layout_model: str = _DEFAULT_LAYOUT_MODEL,
    layout_threshold: float = 0.3,
    layout_device: str | None = None,
    embed_page_image: bool = False,
) -> list[OCRResult]:
    """Concurrently OCR PDFs and images under ``root_folder``.

    At most ``max_concurrency`` files and OCR requests are active. Each PDF
    may process up to ``page_concurrency`` pages concurrently, while returned
    results and Markdown sections preserve deterministic source/page order.
    Existing outputs are omitted from task output. When page progress is shown,
    the newest active file is displayed above aggregate file progress.
    """
    selected_provider = _resolve_provider(provider)
    selected_layout_mode = _resolve_layout_mode(layout_mode)
    selected_model = _resolve_model(selected_provider, model)
    _resolve_prompt(selected_provider, prompt)
    root = _resolve_root(root_folder)
    if dpi <= 0:
        raise ValueError("dpi must be greater than zero")
    if request_timeout_s <= 0:
        raise ValueError("request_timeout_s must be greater than zero")
    if max_concurrency <= 0:
        raise ValueError("max_concurrency must be greater than zero")
    if page_concurrency <= 0:
        raise ValueError("page_concurrency must be greater than zero")
    if not 0 <= layout_threshold <= 1:
        raise ValueError("layout_threshold must be between 0 and 1")
    if selected_layout_mode == "pp-doclayout" and not layout_model.strip():
        raise ValueError("layout_model must not be empty")

    jobs = _ocr_jobs(root)
    if not jobs:
        print(f"No PDF, PNG, or JPEG files found under {root}")
        return []

    results_by_index: dict[int, OCRResult] = {}
    pending_jobs: list[tuple[int, Path, Path]] = []
    for index, (source_path, markdown_path) in enumerate(jobs):
        if not overwrite and markdown_path.is_file():
            results_by_index[index] = OCRResult(source_path, markdown_path, "skipped")
        else:
            pending_jobs.append((index, source_path, markdown_path))

    if not pending_jobs:
        results = [results_by_index[index] for index in range(len(jobs))]
        print(f"OCR complete: 0 processed, {len(results)} skipped, 0 failed")
        return results

    ocr_client = (
        client
        if client is not None
        else _create_async_ocr_client(
            selected_provider,
            host=host,
            request_timeout_s=request_timeout_s,
        )
    )
    if selected_provider == "ollama":
        try:
            await ocr_client.show(selected_model)  # type: ignore[union-attr]
        except Exception as error:
            if client is None:
                await ocr_client.close()
            raise RuntimeError(
                f"Cannot use Ollama model '{selected_model}' at {host}. "
                f"Ensure Ollama is running and run `ollama pull {selected_model}`. "
                f"Original error: {error}"
            ) from error

    active_layout_detector = layout_detector
    owned_layout_detector = False
    if selected_layout_mode == "pp-doclayout" and active_layout_detector is None:
        try:
            active_layout_detector = await asyncio.to_thread(
                create_pp_doclayout_detector,
                model_name=layout_model,
                threshold=layout_threshold,
                device=layout_device,
            )
            owned_layout_detector = True
        except Exception:
            if client is None:
                await ocr_client.close()
            raise

    description = (
        f"OCR files ({selected_provider}, layout={selected_layout_mode}, "
        f"files/requests={max_concurrency}, "
        f"pages/PDF={page_concurrency})"
    )
    progress = _OCRFolderProgress(len(pending_jobs), description, show_page_progress)
    file_semaphore = asyncio.Semaphore(max_concurrency)
    request_semaphore = asyncio.Semaphore(max_concurrency)
    layout_semaphore = asyncio.Semaphore(1)

    async def run_job(
        index: int, source_path: Path, markdown_path: Path
    ) -> tuple[int, OCRResult, float]:
        async with file_semaphore:
            started_at = perf_counter()
            is_pdf = source_path.suffix.casefold() == ".pdf"
            progress.start_file(index, source_path, None if is_pdf else 1)

            def report_pages_total(pages_total: int) -> None:
                progress.set_pages_total(index, pages_total)

            def report_page(
                pages_completed: int, pages_total: int, elapsed_s: float
            ) -> None:
                progress.complete_page(index, pages_completed, pages_total, elapsed_s)

            common_arguments = {
                "client": ocr_client,
                "provider": selected_provider,
                "model": selected_model,
                "prompt": prompt,
                "overwrite": overwrite,
                "request_semaphore": request_semaphore,
                "layout_mode": selected_layout_mode,
                "layout_detector": active_layout_detector,
                "layout_model": layout_model,
                "layout_threshold": layout_threshold,
                "layout_device": layout_device,
                "embed_page_image": embed_page_image,
                "layout_semaphore": layout_semaphore,
            }
            try:
                if is_pdf:
                    result = await _ocr_pdf_async(
                        source_path,
                        markdown_path,
                        dpi=dpi,
                        page_concurrency=page_concurrency,
                        page_started=(
                            report_pages_total if show_page_progress else None
                        ),
                        page_progress=report_page if show_page_progress else None,
                        **common_arguments,
                    )
                else:
                    result = await _ocr_image_async(
                        source_path,
                        markdown_path,
                        **common_arguments,
                    )
                    if result.status == "processed" and show_page_progress:
                        progress.complete_page(index, 1, 1, perf_counter() - started_at)
            except Exception as error:
                result = OCRResult(
                    source_path,
                    markdown_path,
                    "failed",
                    error=f"{type(error).__name__}: {error}",
                )
            finally:
                progress.finish_file(index)
            return index, result, perf_counter() - started_at

    tasks = [
        asyncio.create_task(run_job(index, source_path, markdown_path))
        for index, source_path, markdown_path in pending_jobs
    ]
    completion_records: list[str] = []
    try:
        for completed_task in asyncio.as_completed(tasks):
            index, result, elapsed_s = await completed_task
            results_by_index[index] = result
            if result.status != "skipped":
                error_text = f" | error={result.error}" if result.error else ""
                completion_records.append(
                    f"OCR task: {result.source_path} | status={result.status} | "
                    f"elapsed={elapsed_s:.2f}s{error_text}"
                )
    finally:
        for task in tasks:
            if not task.done():
                task.cancel()
        await asyncio.gather(*tasks, return_exceptions=True)
        progress.close()
        if owned_layout_detector and active_layout_detector is not None:
            try:
                await asyncio.to_thread(active_layout_detector.stop)
            except Exception:
                pass
        if client is None:
            await ocr_client.close()

    for record in completion_records:
        print(record)

    results = [results_by_index[index] for index in range(len(jobs))]

    counts = Counter(result.status for result in results)
    print(
        f"OCR complete: {counts['processed']} processed, "
        f"{counts['skipped']} skipped, {counts['failed']} failed"
    )
    return results

## Configuration and batch execution

Set `OCR_PROVIDER` to `"ollama"` or `"dashscope"`, set `PDF_ROOT` to the root containing PDFs and/or images, then run the final cell with `await`. Set `OCR_LAYOUT_MODE = "pp-doclayout"` for structured reconstruction; leave it as `"plain"` for fast whole-page transcription. Structured mode reuses one detector for the folder run, serializes detector inference for thread safety, and retains region OCR concurrency behind the same global request limit. It creates Markdown, `.layout.json`, and `.assets/` siblings; the Markdown file is exposed last so failed documents do not appear complete. `LAYOUT_DEVICE = None` auto-selects CUDA when available, while `"cpu"` reserves the GPU for Ollama. `EMBED_PAGE_IMAGE` adds a full-page visual fallback at the cost of output size. The configuration reads `OLLAMA_MAX_CONCURRENCY` and `OLLAMA_PAGE_CONCURRENCY` for Ollama, or `OPENAILIKED_MAX_CONCURRENCY` and `OPENAILIKED_PAGE_CONCURRENCY` for DashScope, from `PROJ_ROOT/.env`; existing process-environment values take precedence. `MAX_CONCURRENCY` bounds both active files and all in-flight provider requests. `PAGE_CONCURRENCY` controls concurrent pages within each PDF; set it to `1` for sequential pages. Every request shares the folder-wide limit, and Markdown page order remains deterministic. With `SHOW_PAGE_PROGRESS = True`, a fixed top bar follows the newest active file's completed pages and a second bar tracks all files; completed task records are printed below the bars after processing. Existing Markdown targets are omitted from progress but remain represented as `skipped` results. Ollama requires `ollama pull glm-ocr`. DashScope reads `DASHSCOPE_API_KEY`, `DASHSCOPE_API_URL`, and optional `OPENAILIKED_OCR_MODEL`; its model defaults to `qwen3.7-plus`. Set `OVERWRITE = True` when changing provider or layout mode for an existing target.

In [ ]:
PDF_ROOT = Path("../res/PDF-20260721")  # Root containing PDFs and/or images.
OCR_PROVIDER: OCRProvider = "ollama"  # Change to "dashscope" for Aliyun.
OLLAMA_HOST = "http://127.0.0.1:11434"
OCR_MODEL: str | None = None  # Use the provider or environment default.
OCR_DPI = 200
REQUEST_TIMEOUT_S = 180
OCR_LAYOUT_MODE: OCRLayoutMode = "plain"  # Use "pp-doclayout" for structure.
LAYOUT_MODEL = "PaddlePaddle/PP-DocLayoutV3_safetensors"
LAYOUT_THRESHOLD = 0.3
LAYOUT_DEVICE: str | None = None  # Set to "cpu" to reserve GPU for Ollama.
EMBED_PAGE_IMAGE = False


def _positive_env_int(name: str, default: int) -> int:
    raw_value = os.getenv(name, str(default)).strip()
    try:
        value = int(raw_value)
    except ValueError as error:
        raise ValueError(f"{name} must be a positive integer, got {raw_value!r}") from error
    if value <= 0:
        raise ValueError(f"{name} must be greater than zero, got {value}")
    return value


_CONCURRENCY_ENV_PREFIX = (
    "OLLAMA" if OCR_PROVIDER == "ollama" else "OPENAILIKED"
)
_MAX_CONCURRENCY_DEFAULT = 4 if OCR_PROVIDER == "ollama" else 8
MAX_CONCURRENCY = _positive_env_int(
    f"{_CONCURRENCY_ENV_PREFIX}_MAX_CONCURRENCY",
    _MAX_CONCURRENCY_DEFAULT,
)
PAGE_CONCURRENCY = _positive_env_int(
    f"{_CONCURRENCY_ENV_PREFIX}_PAGE_CONCURRENCY",
    2,
)
SHOW_PAGE_PROGRESS = True
OVERWRITE = False

In [ ]:
# | notest
results = await ocr_folder(
    PDF_ROOT,
    provider=OCR_PROVIDER,
    host=OLLAMA_HOST,
    model=OCR_MODEL,
    dpi=OCR_DPI,
    overwrite=OVERWRITE,
    request_timeout_s=REQUEST_TIMEOUT_S,
    max_concurrency=MAX_CONCURRENCY,
    page_concurrency=PAGE_CONCURRENCY,
    show_page_progress=SHOW_PAGE_PROGRESS,
    layout_mode=OCR_LAYOUT_MODE,
    layout_model=LAYOUT_MODEL,
    layout_threshold=LAYOUT_THRESHOLD,
    layout_device=LAYOUT_DEVICE,
    embed_page_image=EMBED_PAGE_IMAGE,
)

# | notest
results

## Tests

The tests use temporary PDFs/images and fake Ollama/OpenAI-compatible clients, so they do not require a running service, incur cloud charges, or write to the repository.

In [ ]:
# | hide
from contextlib import redirect_stderr, redirect_stdout
from io import StringIO
from types import SimpleNamespace
from tempfile import TemporaryDirectory
from unittest.mock import patch

from fastcore.test import test_eq, test_fail


class FakeOllamaClient:
    def __init__(self, responses=(), show_error: Exception | None = None):
        self.responses = list(responses)
        self.show_error = show_error
        self.show_calls = []
        self.chat_calls = []

    def show(self, model):
        self.show_calls.append(model)
        if self.show_error is not None:
            raise self.show_error
        return {}

    def chat(self, **kwargs):
        self.chat_calls.append(kwargs)
        if not self.responses:
            raise AssertionError("Unexpected chat call")
        response = self.responses.pop(0)
        if isinstance(response, Exception):
            raise response
        return SimpleNamespace(message=SimpleNamespace(content=response))


class FakeOpenAIClient:
    def __init__(self, responses=()):
        self.responses = list(responses)
        self.calls = []
        self.chat = SimpleNamespace(
            completions=SimpleNamespace(create=self._create)
        )

    def _create(self, **kwargs):
        self.calls.append(kwargs)
        if not self.responses:
            raise AssertionError("Unexpected chat completion call")
        response = self.responses.pop(0)
        if isinstance(response, Exception):
            raise response
        if not isinstance(response, str):
            return response
        return SimpleNamespace(
            choices=[SimpleNamespace(message=SimpleNamespace(content=response))]
        )


class FakeAsyncOllamaClient:
    def __init__(
        self, responses=(), show_error: Exception | None = None, delay_s=0.01
    ):
        self.responses = list(responses)
        self.show_error = show_error
        self.delay_s = delay_s
        self.show_calls = []
        self.chat_calls = []
        self.active_calls = 0
        self.max_active_calls = 0
        self.closed = False

    async def show(self, model):
        self.show_calls.append(model)
        if self.show_error is not None:
            raise self.show_error
        return {}

    async def chat(self, **kwargs):
        self.chat_calls.append(kwargs)
        if not self.responses:
            raise AssertionError("Unexpected chat call")
        response = self.responses.pop(0)
        delay_s = self.delay_s
        if isinstance(response, tuple):
            delay_s, response = response
        self.active_calls += 1
        self.max_active_calls = max(self.max_active_calls, self.active_calls)
        try:
            await asyncio.sleep(delay_s)
        finally:
            self.active_calls -= 1
        if isinstance(response, Exception):
            raise response
        return SimpleNamespace(message=SimpleNamespace(content=response))

    async def close(self):
        self.closed = True


class FakeAsyncOpenAIClient:
    def __init__(self, responses=(), delay_s=0.01):
        self.responses = list(responses)
        self.delay_s = delay_s
        self.calls = []
        self.active_calls = 0
        self.max_active_calls = 0
        self.closed = False
        self.chat = SimpleNamespace(
            completions=SimpleNamespace(create=self._create)
        )

    async def _create(self, **kwargs):
        self.calls.append(kwargs)
        if not self.responses:
            raise AssertionError("Unexpected chat completion call")
        response = self.responses.pop(0)
        self.active_calls += 1
        self.max_active_calls = max(self.max_active_calls, self.active_calls)
        try:
            await asyncio.sleep(self.delay_s)
        finally:
            self.active_calls -= 1
        if isinstance(response, Exception):
            raise response
        if not isinstance(response, str):
            return response
        return SimpleNamespace(
            choices=[SimpleNamespace(message=SimpleNamespace(content=response))]
        )

    async def close(self):
        self.closed = True


class FakeLayoutDetector:
    def __init__(self, regions):
        self.regions = regions
        self.calls = []
        self.stopped = False

    def process(self, images, save_visualization=False):
        self.calls.append((images, save_visualization))
        return [self.regions], {}

    def stop(self):
        self.stopped = True


async def assert_async_fails(awaitable, contains: str):
    try:
        await awaitable
    except Exception as error:
        assert contains.casefold() in str(error).casefold()
    else:
        raise AssertionError("Expected awaitable to fail")


def make_pdf(path: Path, labels=("page",), password: str | None = None):
    document = pymupdf.open()
    for label in labels:
        page = document.new_page()
        page.insert_text((72, 72), label)
    if password is None:
        document.save(path)
    else:
        document.save(
            path,
            encryption=pymupdf.PDF_ENCRYPT_AES_256,
            owner_pw="owner-password",
            user_pw=password,
        )
    document.close()


def make_image(path: Path, label: str = "image"):
    document = pymupdf.open()
    page = document.new_page(width=240, height=120)
    page.insert_text((24, 60), label)
    page.get_pixmap(alpha=False).save(path)
    document.close()

In [ ]:
# | hide
def test_source_discovery_and_mapping():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory).resolve()
        (root / "nested").mkdir()
        (root / ".md").mkdir()
        make_pdf(root / "B.PDF")
        make_pdf(root / "nested" / "a.pdf")
        make_image(root / "nested" / "C.PNG")
        make_image(root / "photo.JpEg")
        make_pdf(root / ".md" / "ignored.pdf")
        make_image(root / ".md" / "ignored.jpg")
        (root / "notes.txt").write_text("not a PDF", encoding="utf-8")

        jobs = _ocr_jobs(root)
        test_eq(
            [source.relative_to(root).as_posix() for source, _ in jobs],
            ["B.PDF", "nested/a.pdf", "nested/C.PNG", "photo.JpEg"],
        )
        test_eq(
            [target.relative_to(root).as_posix() for _, target in jobs],
            [".md/B.md", ".md/nested/a.md", ".md/nested/C.md", ".md/photo.md"],
        )


def test_output_collision_is_rejected():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory).resolve()
        make_pdf(root / "same.pdf")
        make_image(root / "same.PNG")
        test_fail(lambda: _ocr_jobs(root), contains="output collision")


test_source_discovery_and_mapping()
test_output_collision_is_rejected()

In [ ]:
# | hide
def test_ocr_pdf_writes_ordered_markdown():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        pdf_path = root / "two-pages.pdf"
        markdown_path = root / ".md" / "two-pages.md"
        make_pdf(pdf_path, ("first", "second"))
        client = FakeOllamaClient(("# First", "Second"))

        result = ocr_pdf(pdf_path, markdown_path, client=client)

        test_eq(result.status, "processed")
        test_eq(result.pages_total, 2)
        test_eq(result.pages_completed, 2)
        test_eq(
            markdown_path.read_text(encoding="utf-8"),
            "<!-- Page 1 -->\n\n# First\n\n<!-- Page 2 -->\n\nSecond\n",
        )
        test_eq(len(client.chat_calls), 2)
        for call in client.chat_calls:
            test_eq(call["model"], "glm-ocr")
            test_eq(call["options"], {"temperature": 0})
            test_eq(call["messages"][0]["content"], "Text Recognition:")
            assert isinstance(call["messages"][0]["images"][0], bytes)


def test_skip_and_overwrite():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        pdf_path = root / "one.pdf"
        markdown_path = root / "one.md"
        make_pdf(pdf_path)
        markdown_path.write_text("existing", encoding="utf-8")

        skipped_client = FakeOllamaClient()
        skipped = ocr_pdf(pdf_path, markdown_path, client=skipped_client)
        test_eq(skipped.status, "skipped")
        test_eq(skipped_client.chat_calls, [])
        test_eq(markdown_path.read_text(encoding="utf-8"), "existing")

        overwritten = ocr_pdf(
            pdf_path,
            markdown_path,
            client=FakeOllamaClient(("replacement",)),
            overwrite=True,
        )
        test_eq(overwritten.status, "processed")
        assert "replacement" in markdown_path.read_text(encoding="utf-8")


test_ocr_pdf_writes_ordered_markdown()
test_skip_and_overwrite()

In [ ]:
# | hide
def test_ocr_image_writes_markdown_for_both_providers():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        png_path = root / "scan.png"
        png_markdown = root / "scan.md"
        make_image(png_path, "local image")
        ollama_client = FakeOllamaClient(("# Local image",))

        local = ocr_image(png_path, png_markdown, client=ollama_client)

        test_eq(local.status, "processed")
        test_eq(local.source_path, png_path.resolve())
        test_eq((local.pages_total, local.pages_completed), (1, 1))
        test_eq(
            png_markdown.read_text(encoding="utf-8"),
            "<!-- Image -->\n\n# Local image\n",
        )
        image_bytes = ollama_client.chat_calls[0]["messages"][0]["images"][0]
        assert image_bytes.startswith(b"\x89PNG\r\n\x1a\n")

        jpg_path = root / "photo.jpg"
        jpg_markdown = root / "photo.md"
        make_image(jpg_path, "cloud image")
        dashscope_client = FakeOpenAIClient(("Cloud image",))
        cloud = ocr_image(
            jpg_path,
            jpg_markdown,
            client=dashscope_client,
            provider="dashscope",
        )

        test_eq(cloud.status, "processed")
        data_url = dashscope_client.calls[0]["messages"][0]["content"][0]["image_url"]["url"]
        assert data_url.startswith("data:image/png;base64,")
        assert base64.b64decode(data_url.split(",", 1)[1]).startswith(b"\x89PNG")


def test_pillow_fallback_decodes_transparent_png():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        image_path = root / "keyshot-style.png"
        markdown_path = root / "keyshot-style.md"
        image = Image.new("RGBA", (4, 3), (20, 40, 60, 0))
        image.putpixel((1, 1), (0, 0, 0, 255))
        image.save(image_path)
        client = FakeOllamaClient(("Fallback image",))
        with patch(
            __name__ + "._load_image_pixmap_with_pymupdf",
            side_effect=RuntimeError("simulated MuPDF PNG decoder failure"),
        ):
            result = ocr_image(image_path, markdown_path, client=client)

        assert result.status == "processed", result.error
        sent_png = client.chat_calls[0]["messages"][0]["images"][0]
        decoded = pymupdf.Pixmap(sent_png)
        test_eq((decoded.width, decoded.height, decoded.alpha), (4, 3, 0))
        test_eq(decoded.samples[:3], b"\xff\xff\xff")


async def test_folder_processes_mixed_pdfs_and_images():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        make_pdf(root / "document.pdf")
        make_image(root / "scan.png")
        client = FakeAsyncOllamaClient(("PDF text", "Image text"))

        captured_output = StringIO()
        captured_progress = StringIO()
        with (
            patch(__name__ + "._running_in_notebook", return_value=False),
            redirect_stdout(captured_output),
            redirect_stderr(captured_progress),
        ):
            results = await ocr_folder(root, client=client, max_concurrency=2)

        test_eq([result.status for result in results], ["processed", "processed"])
        test_eq([result.source_path.name for result in results], ["document.pdf", "scan.png"])
        assert (root / ".md" / "document.md").is_file()
        assert (root / ".md" / "scan.md").is_file()
        test_eq(client.show_calls, ["glm-ocr"])
        test_eq(client.max_active_calls, 2)
        timing_output = captured_output.getvalue()
        for source_name in ("document.pdf", "scan.png"):
            source_path = (root / source_name).resolve()
            assert f"OCR task: {source_path} | status=processed | elapsed=" in timing_output
        progress_output = captured_progress.getvalue()
        assert "OCR pages:" in progress_output
        assert "OCR files (ollama" in progress_output
        assert str((root / "scan.png").resolve()) in progress_output


async def test_folder_hides_skipped_files_from_progress():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        skipped_source = root / "already-done.pdf"
        processed_source = root / "new.pdf"
        make_pdf(skipped_source)
        make_pdf(processed_source)
        skipped_target = root / ".md" / "already-done.md"
        skipped_target.parent.mkdir()
        skipped_target.write_text("existing", encoding="utf-8")
        client = FakeAsyncOllamaClient(("New text",))

        captured_output = StringIO()
        captured_progress = StringIO()
        with (
            patch(__name__ + "._running_in_notebook", return_value=False),
            redirect_stdout(captured_output),
            redirect_stderr(captured_progress),
        ):
            results = await ocr_folder(root, client=client, max_concurrency=2)

        test_eq([result.status for result in results], ["skipped", "processed"])
        combined_output = captured_output.getvalue() + captured_progress.getvalue()
        assert str(skipped_source.resolve()) not in combined_output
        assert str(processed_source.resolve()) in combined_output
        assert "1 skipped" in captured_output.getvalue()


def test_notebook_progress_renders_page_and_file_rows():
    updates: list[str] = []

    class FakeDisplayHandle:
        def update(self, content):
            updates.append(content.data)

    def fake_display(content, *, display_id):
        test_eq(display_id, True)
        updates.append(content.data)
        return FakeDisplayHandle()

    with (
        patch(__name__ + "._running_in_notebook", return_value=True),
        patch("IPython.display.display", side_effect=fake_display),
    ):
        progress = _OCRFolderProgress(2, "OCR files (test)", True)
        progress.start_file(0, Path("older.pdf"))
        progress.set_pages_total(0, 4)
        progress.start_file(1, Path("newest.pdf"), 2)
        progress.complete_page(1, 1, 2, 0.25)
        live_html = updates[-1]
        progress.finish_file(1)
        progress.finish_file(0)
        progress.close()

    assert "OCR pages: newest.pdf" in live_html
    assert "OCR files (test)" in live_html
    assert live_html.index("OCR pages: newest.pdf") < live_html.index(
        "OCR files (test)"
    )
    assert "1/2" in live_html
    assert "<progress" in live_html


async def test_pdf_pages_run_concurrently_and_keep_markdown_order():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        pdf_path = root / "multipage.pdf"
        make_pdf(pdf_path, labels=("one", "two", "three", "four"))
        responses = (
            (0.04, "First"),
            (0.01, "Second"),
            (0.02, "Third"),
            (0.01, "Fourth"),
        )
        client = FakeAsyncOllamaClient(responses)

        captured_output = StringIO()
        captured_progress = StringIO()
        with (
            patch(__name__ + "._running_in_notebook", return_value=False),
            redirect_stdout(captured_output),
            redirect_stderr(captured_progress),
        ):
            results = await ocr_folder(
                root,
                client=client,
                max_concurrency=3,
                page_concurrency=3,
            )

        test_eq(results[0].status, "processed")
        test_eq((results[0].pages_total, results[0].pages_completed), (4, 4))
        test_eq(client.max_active_calls, 3)
        test_eq(
            (root / ".md" / "multipage.md").read_text(encoding="utf-8"),
            "<!-- Page 1 -->\n\nFirst\n\n"
            "<!-- Page 2 -->\n\nSecond\n\n"
            "<!-- Page 3 -->\n\nThird\n\n"
            "<!-- Page 4 -->\n\nFourth\n",
        )
        progress_output = captured_progress.getvalue()
        assert f"OCR pages: {pdf_path.resolve()}" in progress_output
        assert "4/4" in progress_output
        assert "last=" in progress_output


async def test_request_limit_is_global_across_pdfs():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        make_pdf(root / "a.pdf", labels=("a1", "a2", "a3"))
        make_pdf(root / "b.pdf", labels=("b1", "b2", "b3"))
        client = FakeAsyncOllamaClient(("page",) * 6, delay_s=0.02)

        with redirect_stdout(StringIO()):
            results = await ocr_folder(
                root,
                client=client,
                max_concurrency=2,
                page_concurrency=3,
                show_page_progress=False,
            )

        test_eq([result.status for result in results], ["processed", "processed"])
        test_eq(len(client.chat_calls), 6)
        test_eq(client.max_active_calls, 2)


test_ocr_image_writes_markdown_for_both_providers()
test_pillow_fallback_decodes_transparent_png()
await test_folder_processes_mixed_pdfs_and_images()
await test_folder_hides_skipped_files_from_progress()
test_notebook_progress_renders_page_and_file_rows()
await test_pdf_pages_run_concurrently_and_keep_markdown_order()
await test_request_limit_is_global_across_pdfs()

In [ ]:
# | hide
def test_structured_layout_writes_markdown_assets_and_sidecar():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        image_path = root / "sheet.png"
        markdown_path = root / "sheet.md"
        make_image(image_path, "structured sheet")
        detector = FakeLayoutDetector(
            [
                {"index": 0, "label": "doc_title", "score": 0.99, "bbox_2d": [0, 0, 1000, 250], "task_type": "text"},
                {"index": 1, "label": "table", "score": 0.98, "bbox_2d": [0, 250, 500, 1000], "task_type": "table"},
                {"index": 2, "label": "image", "score": 0.97, "bbox_2d": [500, 250, 1000, 1000], "task_type": "figure"},
            ]
        )
        client = FakeOllamaClient(
            ("SCARA Robot", "<table><tr><td>4 kg</td></tr></table>", "Movement range")
        )

        result = ocr_image(
            image_path,
            markdown_path,
            client=client,
            layout_mode="pp-doclayout",
            layout_detector=detector,
            embed_page_image=True,
        )

        assert result.status == "processed", result.error
        test_eq(
            [call["messages"][0]["content"] for call in client.chat_calls],
            ["Text Recognition:", "Table Recognition:", "Figure Recognition:"],
        )
        markdown = markdown_path.read_text(encoding="utf-8")
        assert "# SCARA Robot" in markdown
        assert "<table><tr><td>4 kg</td></tr></table>" in markdown
        assert "![Image](<sheet.assets/page-0001-region-003-image.png>)" in markdown
        assert "![Original page 1](<sheet.assets/page-0001.png>)" in markdown
        assets = sorted((root / "sheet.assets").glob("*.png"))
        test_eq(len(assets), 3)
        layout = json.loads((root / "sheet.layout.json").read_text(encoding="utf-8"))
        test_eq(layout["layout_model"], "PaddlePaddle/PP-DocLayoutV3_safetensors")
        test_eq(layout["pages"][0]["regions"][0]["bbox"], [0, 0, 240, 30])
        test_eq(
            [region["task_type"] for region in layout["pages"][0]["regions"]],
            ["text", "table", "figure"],
        )
        assert detector.calls[0][0][0].size == (240, 120)
        assert not detector.stopped


def test_structured_dashscope_payload_and_failed_bundle_cleanup():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        image_path = root / "cloud.png"
        make_image(image_path)
        detector = FakeLayoutDetector(
            [{"index": 0, "label": "table", "score": 0.9, "bbox_2d": [0, 0, 1000, 1000], "task_type": "table"}]
        )
        cloud_client = FakeOpenAIClient(("<table></table>",))
        cloud_target = root / "cloud.md"
        result = ocr_image(
            image_path,
            cloud_target,
            client=cloud_client,
            provider="dashscope",
            layout_mode="pp-doclayout",
            layout_detector=detector,
        )
        assert result.status == "processed", result.error
        call = cloud_client.calls[0]
        test_eq(call["extra_body"], {"enable_thinking": False, "vl_high_resolution_images": True})
        test_eq(call["messages"][0]["content"][1]["text"], "qwenvl markdown")

        failed_target = root / "failed.md"
        failed_detector = FakeLayoutDetector(
            [
                {"index": 0, "label": "text", "score": 0.9, "bbox_2d": [0, 0, 500, 1000], "task_type": "text"},
                {"index": 1, "label": "table", "score": 0.9, "bbox_2d": [500, 0, 1000, 1000], "task_type": "table"},
            ]
        )
        failed = ocr_image(
            image_path,
            failed_target,
            client=FakeOllamaClient(("text", RuntimeError("region failed"))),
            layout_mode="pp-doclayout",
            layout_detector=failed_detector,
        )
        test_eq(failed.status, "failed")
        assert not failed_target.exists()
        assert not (root / "failed.layout.json").exists()
        assert not (root / "failed.assets").exists()
        assert not list(root.glob(".failed.layout-*"))


async def test_folder_reuses_injected_layout_detector():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        make_image(root / "a.png")
        make_image(root / "b.png")
        detector = FakeLayoutDetector(
            [{"index": 0, "label": "text", "score": 0.9, "bbox_2d": [0, 0, 1000, 1000], "task_type": "text"}]
        )
        client = FakeAsyncOllamaClient(("A", "B"))
        with redirect_stdout(StringIO()):
            results = await ocr_folder(
                root,
                client=client,
                layout_mode="pp-doclayout",
                layout_detector=detector,
                max_concurrency=2,
                show_page_progress=False,
            )
        test_eq([result.status for result in results], ["processed", "processed"])
        test_eq(len(detector.calls), 2)
        test_eq(
            [call["messages"][0]["content"] for call in client.chat_calls],
            ["Text Recognition:", "Text Recognition:"],
        )
        assert (root / ".md" / "a.layout.json").is_file()
        assert (root / ".md" / "b.layout.json").is_file()
        assert not detector.stopped


test_structured_layout_writes_markdown_assets_and_sidecar()
test_structured_dashscope_payload_and_failed_bundle_cleanup()
await test_folder_reuses_injected_layout_detector()

In [ ]:
# | hide
def test_provider_and_model_resolution():
    assert (PROJ_ROOT / "pyproject.toml").is_file()
    test_eq(_resolve_provider("OLLAMA"), "ollama")
    test_fail(lambda: _resolve_provider("unknown"), contains="provider must")
    test_eq(_resolve_layout_mode("PP-DOCLAYOUT"), "pp-doclayout")
    test_fail(lambda: _resolve_layout_mode("unknown"), contains="layout_mode")
    test_eq(_resolve_model("ollama", None), "glm-ocr")
    test_fail(lambda: _resolve_model("ollama", "  "), contains="model must")
    test_eq(_resolve_prompt("dashscope", " custom "), "custom")

    with patch.dict(os.environ, {"OPENAILIKED_OCR_MODEL": ""}, clear=False):
        test_eq(_resolve_model("dashscope", None), "qwen3.7-plus")
    with patch.dict(
        os.environ,
        {"OPENAILIKED_OCR_MODEL": "environment-model"},
        clear=False,
    ):
        test_eq(_resolve_model("dashscope", None), "environment-model")
        test_eq(_resolve_model("dashscope", "explicit-model"), "explicit-model")


test_provider_and_model_resolution()

In [ ]:
# | hide
def test_dashscope_ocr_payload_and_response():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        pdf_path = root / "cloud.pdf"
        markdown_path = root / "cloud.md"
        make_pdf(pdf_path, ("cloud OCR",))
        client = FakeOpenAIClient(("# Cloud OCR",))
        with patch.dict(
            os.environ, {"OPENAILIKED_OCR_MODEL": ""}, clear=False
        ):
            result = ocr_pdf(
                pdf_path,
                markdown_path,
                client=client,
                provider="dashscope",
            )

        test_eq(result.status, "processed")
        test_eq(len(client.calls), 1)
        call = client.calls[0]
        test_eq(call["model"], "qwen3.7-plus")
        test_eq(call["temperature"], 0)
        test_eq(call["extra_body"], {"enable_thinking": False})
        content = call["messages"][0]["content"]
        image_url = content[0]["image_url"]["url"]
        assert image_url.startswith("data:image/png;base64,")
        assert base64.b64decode(image_url.split(",", 1)[1]).startswith(
            b"\x89PNG\r\n\x1a\n"
        )
        assert "Markdown" in content[1]["text"]
        assert "# Cloud OCR" in markdown_path.read_text(encoding="utf-8")


test_dashscope_ocr_payload_and_response()

In [ ]:
# | hide
class CompressiblePixmap:
    def tobytes(self, output="png", jpg_quality=95):
        return b"x" * (32 if output == "png" else 4)


class OversizedPixmap:
    def tobytes(self, output="png", jpg_quality=95):
        return b"x" * 32


def test_dashscope_image_size_fallback_and_failure():
    with patch.dict(globals(), {"_DASHSCOPE_MAX_DATA_URL_BYTES": 60}):
        data_url = _dashscope_image_data_url(CompressiblePixmap())
        assert data_url.startswith("data:image/jpeg;base64,")
        test_fail(
            lambda: _dashscope_image_data_url(OversizedPixmap()),
            contains="10 MiB",
        )


test_dashscope_image_size_fallback_and_failure()

In [ ]:
# | hide
def test_failures_do_not_publish_markdown():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        pdf_path = root / "empty-response.pdf"
        markdown_path = root / "empty-response.md"
        make_pdf(pdf_path)
        result = ocr_pdf(pdf_path, markdown_path, client=FakeOllamaClient(("  ",)))
        test_eq(result.status, "failed")
        assert "empty response" in (result.error or "")
        assert not markdown_path.exists()
        assert not list(root.glob("*.tmp"))


def test_corrupt_image_fails_without_markdown():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        image_path = root / "corrupt.png"
        markdown_path = root / "corrupt.md"
        image_path.write_bytes(b"not an image")

        result = ocr_image(image_path, markdown_path, client=FakeOllamaClient())

        test_eq(result.status, "failed")
        assert "Could not decode image" in (result.error or "")
        assert not markdown_path.exists()


def test_corrupt_and_encrypted_pdfs_fail_cleanly():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        corrupt_pdf = root / "corrupt.pdf"
        corrupt_pdf.write_bytes(b"not a PDF")
        encrypted_pdf = root / "encrypted.pdf"
        make_pdf(encrypted_pdf, password="secret")

        corrupt = ocr_pdf(corrupt_pdf, root / "corrupt.md", client=FakeOllamaClient())
        encrypted = ocr_pdf(encrypted_pdf, root / "encrypted.md", client=FakeOllamaClient())
        test_eq(corrupt.status, "failed")
        test_eq(encrypted.status, "failed")
        assert "password" in (encrypted.error or "").lower()
        assert not (root / "corrupt.md").exists()
        assert not (root / "encrypted.md").exists()


async def test_folder_continues_after_a_failed_pdf():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        make_pdf(root / "a.pdf")
        make_pdf(root / "b.pdf")
        client = FakeAsyncOllamaClient((RuntimeError("model failure"), "# B"))

        results = await ocr_folder(root, client=client, max_concurrency=2)

        test_eq([result.status for result in results], ["failed", "processed"])
        assert not (root / ".md" / "a.md").exists()
        assert (root / ".md" / "b.md").exists()
        test_eq(client.show_calls, ["glm-ocr"])


async def test_concurrent_page_failure_does_not_publish_markdown():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        make_pdf(root / "multipage.pdf", labels=("one", "two", "three"))
        client = FakeAsyncOllamaClient(
            (RuntimeError("page failure"), "Second", "Third")
        )

        with redirect_stdout(StringIO()):
            results = await ocr_folder(
                root,
                client=client,
                max_concurrency=3,
                page_concurrency=3,
                show_page_progress=False,
            )

        test_eq(results[0].status, "failed")
        assert "page failure" in (results[0].error or "")
        test_eq(client.max_active_calls, 3)
        assert not (root / ".md" / "multipage.md").exists()


async def test_dashscope_failures_are_isolated():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        malformed_pdf = root / "malformed.pdf"
        make_pdf(malformed_pdf)
        malformed_target = root / "malformed.md"
        malformed = ocr_pdf(
            malformed_pdf,
            malformed_target,
            client=FakeOpenAIClient((SimpleNamespace(choices=[]),)),
            provider="dashscope",
        )
        test_eq(malformed.status, "failed")
        assert "empty response" in (malformed.error or "")
        assert not malformed_target.exists()

        make_pdf(root / "a.pdf")
        make_pdf(root / "b.pdf")
        client = FakeAsyncOpenAIClient((RuntimeError("cloud failure"), "# B"))
        with patch.dict(
            os.environ, {"OPENAILIKED_OCR_MODEL": ""}, clear=False
        ):
            results = await ocr_folder(
                root, provider="dashscope", client=client, max_concurrency=2
            )
        test_eq(client.max_active_calls, 2)
        test_eq(client.calls[0]["extra_body"], {"enable_thinking": False})
        result_by_name = {result.pdf_path.name: result for result in results}
        test_eq(result_by_name["a.pdf"].status, "failed")
        test_eq(result_by_name["b.pdf"].status, "processed")
        assert not (root / ".md" / "a.md").exists()
        assert (root / ".md" / "b.md").exists()


test_failures_do_not_publish_markdown()
test_corrupt_image_fails_without_markdown()
test_corrupt_and_encrypted_pdfs_fail_cleanly()
await test_folder_continues_after_a_failed_pdf()
await test_concurrent_page_failure_does_not_publish_markdown()
await test_dashscope_failures_are_isolated()

In [ ]:
# | hide
async def test_folder_validation_and_preflight():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        missing = root / "missing"
        await assert_async_fails(ocr_folder(missing), contains="does not exist")

        empty_client = FakeAsyncOllamaClient()
        test_eq(await ocr_folder(root, client=empty_client), [])
        test_eq(empty_client.show_calls, [])

        make_pdf(root / "document.pdf")
        unavailable_client = FakeAsyncOllamaClient(
            show_error=RuntimeError("offline")
        )
        await assert_async_fails(
            ocr_folder(root, client=unavailable_client),
            contains="ollama pull glm-ocr",
        )

        skipped_target = root / ".md" / "document.md"
        skipped_target.parent.mkdir()
        skipped_target.write_text("existing", encoding="utf-8")
        skipped_client = FakeAsyncOllamaClient()
        skipped_output = StringIO()
        with redirect_stdout(skipped_output), redirect_stderr(StringIO()):
            skipped_results = await ocr_folder(root, client=skipped_client)
        test_eq([result.status for result in skipped_results], ["skipped"])
        test_eq(skipped_client.show_calls, [])
        assert str((root / "document.pdf").resolve()) not in skipped_output.getvalue()
        skipped_target.unlink()

        await assert_async_fails(
            ocr_folder(root, provider="invalid"), contains="provider must"
        )
        await assert_async_fails(
            ocr_folder(root, max_concurrency=0), contains="max_concurrency"
        )
        await assert_async_fails(
            ocr_folder(root, page_concurrency=0), contains="page_concurrency"
        )
        await assert_async_fails(
            ocr_folder(root, layout_mode="invalid"), contains="layout_mode"
        )
        await assert_async_fails(
            ocr_folder(root, layout_threshold=1.1), contains="layout_threshold"
        )

        with patch.dict(os.environ, {"DASHSCOPE_API_KEY": ""}, clear=False):
            await assert_async_fails(
                ocr_folder(root, provider="dashscope"),
                contains="DASHSCOPE_API_KEY",
            )
        with patch.dict(
            os.environ,
            {
                "DASHSCOPE_API_KEY": "test-key",
                "DASHSCOPE_API_URL": "https://example.invalid/v1",
            },
            clear=False,
        ):
            await assert_async_fails(
                ocr_folder(root, provider="dashscope"),
                contains="OpenAI-compatible",
            )


await test_folder_validation_and_preflight()

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()

## Restoring tables, figures, lists, and page structure

The current one-request-per-page implementation is optimized for text transcription, not lossless document reconstruction. A representative comparison between [the original SA4A product sheet](<../res/PDF-20260721/01宣传资料/产品样本/单页样本/SA4A 产品单页.jpg>) and [its generated Markdown](<../res/PDF-20260721/.md/01宣传资料/产品样本/单页样本/SA4A 产品单页.md>) shows that the text is mostly present but the document objects and their spatial relationships have been discarded.

| Element | Original | Current Markdown |
|---|---|---|
| Page layout | Two-column product sheet | Flattened text stream |
| Product features | Three-item list | Plain paragraphs |
| Specifications | Grouped three-column table with merged cells | Space-separated lines |
| Figures | Product image and three major engineering-drawing regions | Headings only; no image assets |
| Reading structure | Headings, panels, captions, and footer | Mostly unformatted text |

### Why the structure is lost

1. The Ollama path sends the complete page using only `Text Recognition:`. GLM-OCR exposes separate `Text Recognition:`, `Table Recognition:`, and `Figure Recognition:` modes, so the current request does not perform region-specific recognition. See the [official Ollama GLM-OCR usage](https://ollama.com/library/glm-ocr).
2. The notebook calls the raw model directly and does not run a layout detector. The official GLM-OCR pipeline combines PP-DocLayout-V3 layout analysis, parallel region recognition, and a result formatter that returns Markdown plus JSON layout details. See the [official GLM-OCR SDK](https://github.com/zai-org/GLM-OCR).
3. A chat response can return text or descriptions but cannot return the original figure pixels as new files. Figure regions must be cropped from the source page, saved as assets, and referenced from the Markdown.
4. CommonMark has no representation for columns, absolute coordinates, or merged table cells. Semantic reading order should be represented linearly; exact merged tables require inline HTML, and exact page appearance requires HTML/CSS or an embedded image of the original page.
5. The SA4A image is 5031×3437, approximately 17.29 megapixels, and contains very small engineering annotations. Whole-page visual encoding can downscale those details before recognition. Region crops preserve substantially more effective resolution.

### Recommended reconstruction pipeline

A reliable structured mode should use the original page as the source of truth and retain an intermediate layout representation:

1. Detect page regions and record each region's type, bounding box, page number, confidence, and reading order in a `.layout.json` sidecar.
2. Crop each detected region from the lossless rendered page.
3. Route text, headings, and lists through text recognition; tables through table recognition; and diagrams through figure recognition.
4. Save every non-text region as an image asset even when the model also returns a textual description. Never regenerate engineering drawings from model output.
5. Normalize recognized blocks into semantic objects and assemble them in deterministic reading order.
6. Render simple tables as Markdown and tables containing merged cells as inline HTML `<table>` elements.
7. Optionally embed the complete original page at the top as a visual-fidelity fallback.

For example, a source should produce a sibling asset directory:

```text
SA4A 产品单页.md
SA4A 产品单页.layout.json
SA4A 产品单页.assets/
├── product.png
├── movement-range.png
├── installation-top.png
└── installation-bottom.png
```

The resulting document can then preserve semantic structure and original figure pixels:

```markdown
# SCARA机器人

## SA4A系列

最大可搬运质量 4 kg，可达最大半径 400 mm。

## 产品特点

- **性能卓越：** 全新的自主核心部件及算法……
- **扩展性强：** 本体标配多路气管、信号线及 CAT5E 网口……
- **使用便捷：** 轻量、紧凑化的结构设计……

## 规格参数

<table>
  <tr><th>型号</th><th colspan="2">SA4A-4/0.40</th></tr>
  <tr><th rowspan="3">机械臂长（mm）</th><th>J1+J2</th><td>400</td></tr>
  <tr><th>J1</th><td>225</td></tr>
  <tr><th>J2</th><td>175</td></tr>
</table>

## 运动范围

![运动范围](<SA4A 产品单页.assets/movement-range.png>)

## 安装尺寸

![安装尺寸](<SA4A 产品单页.assets/installation-top.png>)

![本体安装尺寸](<SA4A 产品单页.assets/installation-bottom.png>)
```

### Provider-specific changes

#### Local Ollama and GLM-OCR

Prompt changes alone are insufficient because a full page must first be segmented. The preferred local path is the official GLM-OCR layout pipeline. An alternative that retains the current Ollama service is to run PP-DocLayout separately, then send detected crops with `Text Recognition:`, `Table Recognition:`, or `Figure Recognition:` according to their region types. The notebook must still save figure crops itself.

#### DashScope and Qwen

Replace the generic transcription prompt with the documented `qwenvl markdown` or `qwenvl html` document-parsing prompt. Qwen document parsing can preserve table/image position information. See [DashScope document parsing](https://www.alibabacloud.com/help/en/model-studio/vision#document-parsing).

The OpenAI-compatible request should also enable high-resolution visual processing:

```python
extra_body={
    "enable_thinking": False,
    "vl_high_resolution_images": True,
}
```

Without this flag, Qwen3.7's default visual limit is approximately 2.62 megapixels; high-resolution mode raises the limit to 16,777,216 pixels. The source page is slightly larger than that maximum and will still be downscaled, so layout-based cropping remains necessary for small labels and dimension annotations. See [DashScope high-resolution processing](https://www.alibabacloud.com/help/en/model-studio/vision#process-high-resolution-images).

### Implemented notebook extension

The notebook now implements this design through `layout_mode="pp-doclayout"` while preserving `layout_mode="plain"` as the default fast path. It loads the official GLM-OCR `PPDocLayoutDetector` once per folder run, uses the detector's reading order, sends text/table/formula/figure crops through provider-specific prompts, saves original non-text pixels under `.assets/`, and publishes a UTF-8 `.layout.json` sidecar containing page dimensions, labels, confidence scores, pixel bounding boxes, prompts, assets, and recognized content. DashScope region requests use `qwenvl markdown` with `vl_high_resolution_images=True`. Set `embed_page_image=True` when an exact visual fallback is also required. The optional detector runtime is installed with `uv sync --extra ocr-layout`. Markdown remains a semantic rendering of the retained layout data rather than its only representation.